# 🌾 SAQYA AI (ساقية) — RAG Pipeline for a Farmer Question-Answering Assistant

**Goal:** a farmer asks a question about crops or farming in plain language — **in Arabic or English** —
and gets an answer, in the same language, that is **grounded in our agricultural documents**, with
citations to the file and page it came from. The documents themselves can be Arabic, English or both.

This notebook builds and evaluates the whole retrieval pipeline that the FastAPI backend serves:

| Section | What happens |
|---|---|
| 2.1 Load & Inspect | read every PDF in `data/clean/` (the cleaned corpus) — or `data/raw/` if `clean/` is empty — count pages, flag files that fail or need OCR |
| 2.2 Chunking | clean the text (incl. Arabic fixes) and split it into chunks: section-aware with heading context for the cleaned PDFs, sentence-aware fixed-size otherwise |
| 2.3 Embeddings & Vector Store | embed chunks with multilingual `BAAI/bge-m3` (or its Egyptian-Arabic fine-tune from `egyptian_finetune.ipynb`), store them in a persisted Chroma DB |
| 2.4 Retrieval & Prompting | hybrid search (embeddings + BM25 keywords, both languages), calibrated relevance gate, bilingual prompt, citations, answer self-check, local Ollama LLM |
| 2.5 Vision Component | Core Track — not used (Extended Track idea described) |
| 2.6 Evaluation | Arabic + English test questions, incl. out-of-scope ones → results table and failure analysis |
| 2.7 Export | vector store + `rag_config.json` written to `backend/data/vector_store/` |

**Before running:** clean your PDFs with `notebooks/clean_rag_pdfs_no_ai.ipynb` and put the results in `data/clean/`
(the original downloads can stay in `data/raw/`), optionally run `notebooks/egyptian_finetune.ipynb` so the embeddings
understand Egyptian Arabic (used automatically when `models/bge-m3-egyptian/` exists), start Ollama and pull the model
(`ollama pull qwen2.5:3b`, or `qwen2.5:7b` if your GPU has 8 GB or more), edit
`data/eval_questions.json` so the questions match your documents, then **Kernel → Restart & Run All**.

The retrieval, prompt and answer checks are **imported from the backend** (`backend/app/services/`), so
the numbers in 2.6 measure exactly what the API serves.

## 0. Setup & configuration

In [1]:
import hashlib
import itertools
import json
import os
import re
import sys
import time
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pdfminer.high_level import extract_pages
from pdfminer.layout import LAParams, LTAnno, LTChar, LTContainer, LTTextContainer, LTTextLine
from pypdf import PdfReader
from wordfreq import zipf_frequency

pd.set_option("display.max_colwidth", 100)

# Paths work whether Jupyter was started in the project root or in notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
# Cleaned PDFs (made by notebooks/clean_rag_pdfs_no_ai.ipynb) live in data/clean/; the original downloads
# stay in data/raw/ for reference. The pipeline reads data/clean/ when it holds documents, else data/raw/.
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOCS_DIR = CLEAN_DIR if any(p.suffix.lower() in {".pdf", ".txt", ".md"} for p in CLEAN_DIR.glob("*")) else RAW_DIR
EVAL_FILE = PROJECT_ROOT / "data" / "eval_questions.json"
VECTOR_STORE_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
RESULTS_DIR = PROJECT_ROOT / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the backend's Ollama settings if backend/.env exists
load_dotenv(PROJECT_ROOT / "backend" / ".env")

# ---- pipeline configuration (exported to rag_config.json in 2.7) ----
# Multilingual embedding model: Arabic and English questions/passages share one vector space,
# so an Arabic question can find an English passage and vice versa.
BASE_EMBEDDING_MODEL = "BAAI/bge-m3"
# Egyptian Arabic: notebooks/egyptian_finetune.ipynb fine-tunes bge-m3 on farmers' Egyptian questions (no AI involved)
# and saves it in models/bge-m3-egyptian/. When that folder exists it is used, and its path (relative to the project)
# goes into rag_config.json so the backend loads the same model. Set USE_EGYPTIAN_MODEL = False for the original.
EGYPTIAN_MODEL_DIR = PROJECT_ROOT / "models" / "bge-m3-egyptian"
USE_EGYPTIAN_MODEL = True
EMBEDDING_MODEL = (EGYPTIAN_MODEL_DIR.relative_to(PROJECT_ROOT).as_posix()
                   if USE_EGYPTIAN_MODEL and (EGYPTIAN_MODEL_DIR / "modules.json").exists() else BASE_EMBEDDING_MODEL)


def model_location(name: str) -> str:
    """A local model folder (relative to the project) as a full path; a Hugging Face model id as it is."""
    local = PROJECT_ROOT / name
    return str(local) if local.is_dir() else name


# e5 models were trained with these prefixes; other models use none
PREFIXES = {"intfloat/": ("query: ", "passage: ")}
QUERY_PREFIX, PASSAGE_PREFIX = next(
    (v for k, v in PREFIXES.items() if EMBEDDING_MODEL.startswith(k)), ("", ""))

COLLECTION_NAME = "agri_docs"
CHUNK_SIZE = 800        # characters per chunk (≈200-250 tokens for Arabic, ≈170 for English)
CHUNK_OVERLAP = 150     # characters repeated between neighbouring chunks
MIN_CHUNK_CHARS = 50    # drop stray fragments (lone captions, page debris)
TOP_K = 6               # chunks given to the LLM
# Keyword search (BM25) fused with the embedding search (same as the API's HYBRID_SEARCH)
HYBRID_SEARCH = (os.getenv("HYBRID_SEARCH") or "true").lower() != "false"

# A question whose best cosine similarity is below this is "not about our documents" and is refused
# without calling the LLM. Each model has its own scale (e5 scores even unrelated text around 0.7),
# so we start from a per-model default and calibrate it on the evaluation questions in 2.4.
DEFAULT_THRESHOLDS = {
    "intfloat/multilingual-e5-base": 0.80,
    "intfloat/multilingual-e5-small": 0.80,
    "BAAI/bge-m3": 0.45,
    "models/bge-m3-egyptian": 0.45,   # the fine-tune starts from bge-m3's scale; calibrated below anyway
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": 0.35,
}
MIN_SIMILARITY = DEFAULT_THRESHOLDS.get(EMBEDDING_MODEL, 0.5)
CALIBRATE_THRESHOLD = True

OLLAMA_HOST = os.getenv("OLLAMA_HOST") or "http://localhost:11434"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL") or "qwen2.5:3b"   # handles Arabic and English
OLLAMA_THINK = {"true": True, "false": False}.get((os.getenv("OLLAMA_THINK") or "").lower())
# Also search with the question translated Arabic <-> English (same as the API's QUERY_TRANSLATION)
QUERY_TRANSLATION = (os.getenv("QUERY_TRANSLATION") or "true").lower() != "false"
# Context window for the LLM: Ollama's default is too small for 5 Arabic passages + the rules
OLLAMA_NUM_CTX = int(os.getenv("OLLAMA_NUM_CTX") or 8192)
# Check every answer and let the LLM retry once (same as the API's SELF_CHECK)
SELF_CHECK = (os.getenv("SELF_CHECK") or "true").lower() != "false"

# The prompt template, citation logic and Arabic text helpers live in the backend;
# importing them here means the evaluation tests exactly what the API serves.
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
from app.utils.text import detect_language, normalize_for_match, strip_arabic_marks, to_ascii_digits

print("Project root :", PROJECT_ROOT)
print("Documents    :", DOCS_DIR)
print("Vector store :", VECTOR_STORE_DIR)
print("Embeddings   :", EMBEDDING_MODEL, f"(prefixes {QUERY_PREFIX!r} / {PASSAGE_PREFIX!r})"
      + (" — bge-m3 fine-tuned for Egyptian Arabic" if EMBEDDING_MODEL != BASE_EMBEDDING_MODEL else ""))
print("LLM          :", OLLAMA_MODEL, "@", OLLAMA_HOST)

Project root : D:\agri-rag-assistant
Documents    : D:\agri-rag-assistant\data\clean
Vector store : D:\agri-rag-assistant\backend\data\vector_store
Embeddings   : BAAI/bge-m3 (prefixes '' / '')
LLM          : qwen2.5:3b @ http://localhost:11434


## 2.1 Load & Inspect

Every file in `data/clean/` (or `data/raw/` when `clean/` is empty; PDF, plus `.txt`/`.md` if you have any) is read page by page.
Byte-identical copies of the same PDF are detected (MD5 hash) and only the first is used.
Run `notebooks/data_check.ipynb` first to find garbled, scanned, near-duplicate or off-topic files.

**Choosing the PDF extractor for Arabic.** Many Arabic PDFs draw their glyphs left to right (visual
order), and one glyph often stands for two letters (`لأ`, `تح`, `جم`). We compared four extractors on
the project's Arabic bulletins:

| Extractor | Valid Arabic words | Numbers kept | Problem |
|---|---|---|---|
| `pypdf` | 94–99 % | ✗ | drops numbers on Arabic lines: `من 15 إلى 30 نوفمبر` → `من نوفمبر` |
| PyMuPDF / PDFium (character positions) | 65–95 % | ✓ | splits two-letter glyphs the wrong way: `تحتل` → `حتتل`, `فبراير` → `رفباير` |
| pdfminer, default text | — | ✓ | whole lines come out backwards |
| **pdfminer, glyphs read right-to-left (used here)** | **94–99 %** | **✓** | — |

So each line is read with pdfminer glyph by glyph from right to left; a two-letter glyph keeps its
letters in order, and digits / Latin words are turned back to left-to-right.

Pages are normally read with pdfminer's own block layout (it handles English columns and tables well).
**Arabic pages where one printed line is stored as several touching fragments** that pdfminer files in
different blocks (e.g. `AGRI_013`, many tables) would come out with interleaved sentences, so on those
pages lines are rebuilt from fragment positions instead:
- fragments on the same baseline are joined right to left, keeping the spaces at their edges — tables
  come out row by row (`نترات نشادر | اول يونيو | 15 يونيو | 18 | كجم`);
- if many lines have wide text on both sides of the same vertical gap, the page is read one column at a
  time (right column first); full-width lines such as titles end a column band;
- sideways margin titles (a stack of one-letter fragments) are dropped.

On every Arabic page, mirrored brackets some PDFs store (`)كذا(`) are turned the right way round, and
Arabic words inside English lines are put back in reading order.

If pdfminer fails on a file, pypdf is used as a fallback. `EXTRACTOR = "pypdf"` switches extractors.

A page that yields fewer than 30 characters is almost certainly a scanned image and would need OCR —
those pages are counted, not silently dropped. Each document's language is estimated from the share of
Arabic letters in its text.

**Cleaned PDFs (`data/clean/`).** The corpus is now the output of `clean_rag_pdfs_no_ai.ipynb`: every PDF
was rebuilt with its cover, credits, contents pages, running headers and page numbers removed, Arabic text
in reading order and numbers fixed, and the content laid out in one template (title, numbered sections,
sub-sections, bullet lists, table rows as one line each). Those PDFs are recognised automatically
(`is_structured_pdf`: Chromium / Skia producer) and read **structure-aware**: the title, section and
sub-section headings each have their own font size, so they come out as `#` / `##` / `###` lines, each
paragraph or list item becomes one block, and the generated contents list is skipped. Any other PDF you
drop into the folder is still read with the right-to-left pdfminer reader described below.


In [2]:
SUPPORTED = {".pdf", ".txt", ".md"}
OCR_THRESHOLD = 30  # fewer extracted characters than this -> probably a scanned page
EXTRACTOR = "pdfminer"  # or "pypdf"

ARABIC_CHARS = "؀-ۿݐ-ݿﭐ-﷿ﹰ-﻿"   # incl. presentation forms
_ARABIC_CHAR_RE = re.compile(f"[{ARABIC_CHARS}]")
_LATIN_CHAR_RE = re.compile(r"[A-Za-z]")
_LTR_RUN = r"[A-Za-z0-9٠-٩۰-۹][A-Za-z0-9٠-٩۰-۹.,٫%/:+\-]*"
_LTR_RUNS_RE = re.compile(f"{_LTR_RUN}(?:\\s+{_LTR_RUN})*")
_SWAP_BRACKETS = str.maketrans("()[]", ")(][")


def is_rtl(text: str) -> bool:
    return len(_ARABIC_CHAR_RE.findall(text)) > len(_LATIN_CHAR_RE.findall(text))


def fragment_text(line) -> str:
    """Text of one pdfminer line fragment in reading order.

    Many Arabic PDFs draw their glyphs left to right (visual order). Such fragments are read right to
    left *glyph by glyph*: a ligature glyph that stands for several letters (لأ, تح, جم ...) keeps its
    letters in order, and runs of digits / Latin words are turned back to left-to-right.
    """
    items = [o for o in line if isinstance(o, (LTChar, LTAnno))]
    text = "".join(o.get_text() for o in items)
    if not is_rtl(text):
        return _fix_arabic_runs(items) if _ARABIC_CHAR_RE.search(text) else text
    xs = [o.x0 for o in items if isinstance(o, LTChar)]
    if sum(b > a for a, b in zip(xs, xs[1:])) >= len(xs) / 2:  # glyphs stored left to right
        text = "".join(o.get_text() for o in reversed(items))
        text = _LTR_RUNS_RE.sub(lambda m: m.group(0)[::-1], text)
    return text


def _fix_arabic_runs(items) -> str:
    """English line containing Arabic words: if the Arabic glyphs are stored left to right, reverse each
    Arabic run glyph by glyph (keeps ligature glyphs whole)."""
    def is_ar(o):
        return isinstance(o, LTChar) and bool(_ARABIC_CHAR_RE.search(o.get_text()))
    ar_x = [o.x0 for o in items if is_ar(o)]
    if len(ar_x) < 2 or sum(b > a for a, b in zip(ar_x, ar_x[1:])) < (len(ar_x) - 1) / 2:
        return "".join(o.get_text() for o in items)
    out, run = [], []
    for o in items + [None]:
        if o is not None and (is_ar(o) or (run and o.get_text() == " ")):
            run.append(o.get_text())
            continue
        trailing = []
        while run and run[-1] == " ":
            trailing.append(run.pop())
        out.extend(reversed(run))
        out.extend(trailing)
        run = []
        if o is not None:
            out.append(o.get_text())
    return "".join(out)


def _text_lines(obj):
    if isinstance(obj, LTTextLine):
        yield obj
    elif isinstance(obj, LTContainer):
        for child in obj:
            yield from _text_lines(child)


def _fix_brackets(text: str) -> str:
    """Some PDFs store mirrored brackets: on lines where ')' comes before '(', swap them."""
    def violations(t):
        depth = bad = 0
        for ch in t:
            depth += (ch == "(") - (ch == ")")
            if depth < 0:
                bad, depth = bad + 1, 0
        return bad + depth
    lines = []
    for line in text.split("\n"):  # line by line: one page can mix both conventions
        swapped = line.translate(_SWAP_BRACKETS)
        lines.append(swapped if violations(swapped) < violations(line) else line)
    return "\n".join(lines)


def _block_text(layout) -> str:
    """pdfminer's own reading: text blocks in layout order (handles columns and tables well)."""
    blocks = []
    for element in layout:
        if isinstance(element, LTTextContainer):
            lines = [re.sub(r"\s+", " ", fragment_text(l)).strip() for l in _text_lines(element)]
            lines = [l for l in lines if l]
            if lines:
                blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


def _split_line_pairs(layout) -> tuple[int, int]:
    """Count touching fragments of the same printed line that pdfminer put in different blocks."""
    frags = []
    for b, element in enumerate(layout):
        if isinstance(element, LTTextContainer):
            for line in _text_lines(element):
                if line.get_text().strip():
                    frags.append((b, line))
    frags.sort(key=lambda f: (-round((f[1].y0 + f[1].y1) / 2), f[1].x0))
    pairs = 0
    rows = 1 if frags else 0
    for (b1, l1), (b2, l2) in zip(frags, frags[1:]):
        same_row = abs((l1.y0 + l1.y1) / 2 - (l2.y0 + l2.y1) / 2) <= 0.5 * min(l1.height, l2.height)
        rows += not same_row
        if same_row and b1 != b2 and l2.x0 - l1.x1 < 0.8 * max(l1.height, 1):
            pairs += 1
    return pairs, rows


def page_text_from_layout(layout) -> str:
    """Page text in reading order.

    Normally pdfminer's block reading is used. Some PDFs store one printed line as several touching
    fragments and pdfminer files them in different blocks, which scrambles sentences; on such pages
    the lines are rebuilt from fragment positions instead (right to left for Arabic), and real text
    columns, if any, are read one column at a time.
    """
    block_text = _block_text(layout)
    if not is_rtl(block_text):  # English pages: pdfminer's block reading
        return block_text
    pairs, rows = _split_line_pairs(layout)
    if pairs < max(3, 0.1 * rows):
        return _fix_brackets(block_text)
    frags = []
    for line in _text_lines(layout):
        raw = fragment_text(line).replace("\n", "")
        text = re.sub(r"\s+", " ", raw).strip()
        if text:
            sizes = [c.size for c in line if isinstance(c, LTChar)]
            frags.append({"x0": line.x0, "x1": line.x1, "y": (line.y0 + line.y1) / 2,
                          "h": max(line.height, 1.0), "size": sorted(sizes)[len(sizes) // 2] if sizes else line.height,
                          "text": text, "lead": raw[:1].isspace(), "trail": raw[-1:].isspace()})
    # sideways text (a title running down the margin) arrives as a stack of one-letter fragments: drop it
    def lone_letter(f):
        return len(f["text"]) == 1 and f["text"].isalpha()
    singles = Counter(round(f["x0"] / 3) for f in frags if lone_letter(f))
    frags = [f for f in frags if not (lone_letter(f) and singles[round(f["x0"] / 3)] >= 4)]
    if not frags:
        return ""
    rtl_page = is_rtl(" ".join(f["text"] for f in frags))

    # 1. fragments on the same baseline -> one visual line
    frags.sort(key=lambda f: -f["y"])
    rows = []
    for f in frags:
        if rows and abs(rows[-1]["y"] - f["y"]) <= 0.5 * min(rows[-1]["h"], f["h"]):
            rows[-1]["frags"].append(f)
        else:
            rows.append({"y": f["y"], "h": f["h"], "frags": [f]})

    # 2. within a line, split at wide gaps (columns / table cells) into segments
    for row in rows:
        row["frags"].sort(key=lambda f: f["x0"])
        segments, current = [], [row["frags"][0]]
        for a, b in zip(row["frags"], row["frags"][1:]):
            if b["x0"] - a["x1"] > max(0.8 * a["size"], 5):
                segments.append(current)
                current = [b]
            else:
                current.append(b)
        segments.append(current)
        row["segments"] = segments

    # 3. column gutters: an x position with wide text on both sides in many lines, and few lines crossing it
    width = max(f["x1"] for f in frags) - min(f["x0"] for f in frags)
    candidates = sorted({round((a[-1]["x1"] + b[0]["x0"]) / 2)
                         for r in rows for a, b in zip(r["segments"], r["segments"][1:])})
    gutters = []
    for x in candidates:
        crossing = sum(any(sg[0]["x0"] < x < sg[-1]["x1"] for sg in r["segments"]) for r in rows)
        column_rows = 0
        for r in rows:
            left = [sg for sg in r["segments"] if sg[-1]["x1"] <= x]
            right = [sg for sg in r["segments"] if sg[0]["x0"] >= x]
            if left and right:
                l, rr = max(left, key=lambda sg: sg[-1]["x1"]), min(right, key=lambda sg: sg[0]["x0"])
                column_rows += (l[-1]["x1"] - l[0]["x0"] > 0.2 * width) and (rr[-1]["x1"] - rr[0]["x0"] > 0.2 * width)
        if crossing <= 0.2 * len(rows) and column_rows >= max(4, 0.15 * len(rows)):
            if not gutters or x - gutters[-1] > 0.1 * width:
                gutters.append(x)

    def segment_text(seg) -> str:
        parts = sorted(seg, key=lambda f: -f["x0"]) if rtl_page else seg
        out = parts[0]["text"]
        for prev, f in zip(parts, parts[1:]):
            gap = (prev["x0"] - f["x1"]) if rtl_page else (f["x0"] - prev["x1"])
            space = prev["trail"] or f["lead"] or gap > 0.15 * f["size"]
            out += (" " if space else "") + f["text"]
        return out

    def column_of(seg) -> int:
        mid = (seg[0]["x0"] + seg[-1]["x1"]) / 2
        col = sum(mid > g for g in gutters)
        return (len(gutters) - col) if rtl_page else col

    # 4. emit in reading order; a line that crosses a gutter (title, full-width paragraph) ends a band
    out, band = [], []

    def flush():
        for col in sorted({c for c, _, _ in band}):
            out.extend(t for c, _, t in band if c == col)
            out.append("")
        band.clear()

    for row in rows:
        segs = row["segments"][::-1] if rtl_page else row["segments"]
        if gutters and not any(any(s[0]["x0"] < g < s[-1]["x1"] for g in gutters) for s in segs):
            for s in segs:
                band.append((column_of(s), row["y"], segment_text(s)))
        else:
            flush()
            out.append("   ".join(segment_text(s) for s in segs))
    flush()
    text = re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()
    return _fix_brackets(text) if rtl_page else text


def pdfminer_pages(path: Path) -> list[str]:
    return [page_text_from_layout(layout) for layout in extract_pages(str(path), laparams=LAParams())]


def pypdf_pages(path: Path) -> list[str]:
    reader = PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")  # many "protected" PDFs open with an empty password
    return [page.extract_text() or "" for page in reader.pages]


# ---------------------------------------------------------------- cleaned PDFs (structure-aware reading)
# PDFs rebuilt by notebooks/clean_rag_pdfs_no_ai.ipynb (Chromium output, one template) have reliable
# structure: the title, section and sub-section headings each use their own font size. Reading that
# structure lets every chunk stay inside one section and carry its heading path (see 2.2).
STRUCTURED_SIZES = [(21.0, 1), (16.5, 2), (13.5, 3), (12.4, 4)]  # font size (pt) -> heading level
CONTENTS_TITLES = {"المحتويات", "الفهرس", "contents", "table of contents"}


def is_structured_pdf(path: Path) -> bool:
    """True for PDFs written by the cleaning notebook (Chromium / Skia)."""
    try:
        meta = PdfReader(str(path)).metadata or {}
    except Exception:
        return False
    return "skia" in str(meta.get("/Producer", "")).lower() and "chrom" in str(meta.get("/Creator", "")).lower()


def _fix_number_marker(text: str) -> str:
    """'.2 ميعاد الزراعة' -> '2. ميعاد الزراعة', '-1 الزراعة اليدوية' -> '1- الزراعة اليدوية',
    and on Arabic lines '%97' -> '97%' (the % is read on the wrong side of a right-to-left number)."""
    text = re.sub(r"^\.(\d{1,3})\s+", r"\1. ", text)
    text = re.sub(r"^-(\d{1,3})\s+", r"\1- ", text)
    if is_rtl(text):
        text = re.sub(r"%\s?(\d+(?:\.\d+)?(?:-\d+(?:\.\d+)?)?)", r"\1%", text)
    return text


def structured_pages(path: Path) -> list[str]:
    """Page texts of a cleaned PDF: '# title', '## section', '### sub-section' lines, one paragraph or
    list item per block (blank line between blocks). The contents list at the start is skipped."""
    pages, in_contents, pitches = [], False, []
    layouts = list(extract_pages(str(path), laparams=LAParams()))
    page_rows = []
    for layout in layouts:
        rows = []
        for line in _text_lines(layout):
            sizes = [round(c.size, 1) for c in line if isinstance(c, LTChar)]
            text = re.sub(r"\s+", " ", fragment_text(line)).strip()
            if not sizes or not text:
                continue
            size = Counter(sizes).most_common(1)[0][0]
            level = next((lvl for limit, lvl in STRUCTURED_SIZES if size >= limit), 0)
            rows.append({"top": line.y1, "level": level, "text": text})
        rows.sort(key=lambda r: -r["top"])
        pitches += [a["top"] - b["top"] for a, b in zip(rows, rows[1:]) if a["level"] == b["level"] == 0]
        page_rows.append(rows)
    # body line pitch inside a paragraph; a bigger step means a new paragraph or list item
    pitch = Counter(round(p) for p in pitches if p > 0).most_common(1)[0][0] if pitches else 0
    for rows in page_rows:
        blocks, prev = [], None
        for r in rows:
            text = _fix_number_marker(r["text"])
            if in_contents and (r["level"] or not re.match(r"^\s*(\d+[.\-]|…)", text)):
                in_contents = False           # contents entries are "N. title" lines
            if r["level"]:
                if unicodedata.normalize("NFKC", text).strip(" :").lower() in CONTENTS_TITLES:
                    in_contents = True
                    continue
                same = prev is not None and prev["level"] == r["level"] and prev["top"] - r["top"] < 2.2 * pitch
                if same and blocks:           # a heading wrapped over two lines
                    blocks[-1] += " " + text
                else:
                    blocks.append("#" * r["level"] + " " + text)
            elif in_contents:
                continue
            else:
                new_block = (prev is None or prev["level"] != 0 or not pitch
                             or prev["top"] - r["top"] > 1.1 * pitch)
                if new_block or not blocks:
                    blocks.append(text)
                else:
                    blocks[-1] += " " + text
            prev = r
        pages.append("\n\n".join(blocks))
    return pages


def load_document(path: Path) -> list[dict]:
    """Return [{'page': n, 'text': ...}] with 1-based page numbers."""
    if path.suffix.lower() == ".pdf":
        if is_structured_pdf(path):  # cleaned PDF: headings and blocks kept
            return [{"page": i, "text": t} for i, t in enumerate(structured_pages(path), 1)]
        try:
            texts = pdfminer_pages(path) if EXTRACTOR == "pdfminer" else pypdf_pages(path)
        except Exception:  # damaged or unusual PDF: fall back to pypdf
            texts = pypdf_pages(path)
        return [{"page": i, "text": t} for i, t in enumerate(texts, 1)]
    return [{"page": 1, "text": path.read_text(encoding="utf-8", errors="ignore")}]


def file_md5(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()


files = sorted(p for p in DOCS_DIR.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED)
ignored = sorted(p.name for p in DOCS_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in SUPPORTED | {""})
if not files:
    raise FileNotFoundError(
        f"No documents found in {DOCS_DIR}. Copy your cleaned PDFs to data/clean/ (or the originals to "
        "data/raw/) and run the notebook again."
    )
if ignored:
    print(f"Ignored (not PDF/TXT/MD): {ignored}")

raw_pages, inventory, failed, duplicates = [], [], [], []
seen_hashes = {}
for path in files:
    rel = path.relative_to(DOCS_DIR).as_posix()
    digest = file_md5(path)
    if digest in seen_hashes:  # byte-identical copy: indexing it twice would duplicate every citation
        duplicates.append({"file": rel, "same_as": seen_hashes[digest]})
        continue
    seen_hashes[digest] = rel
    try:
        pages = load_document(path)
    except Exception as exc:
        failed.append({"file": rel, "error": f"{type(exc).__name__}: {exc}"})
        continue
    chars = [len(p["text"].strip()) for p in pages]
    ocr_pages = [p["page"] for p, c in zip(pages, chars) if c < OCR_THRESHOLD]
    page_langs = Counter(detect_language(unicodedata.normalize("NFKC", p["text"]))
                         for p, c in zip(pages, chars) if c >= OCR_THRESHOLD)
    language = ("mixed" if len(page_langs) > 1 and min(page_langs.values()) / sum(page_langs.values()) > 0.2
                else (page_langs.most_common(1)[0][0] if page_langs else "unknown"))
    inventory.append({
        "file": rel,
        "format": path.suffix.lower().lstrip("."),
        "structured": path.suffix.lower() == ".pdf" and is_structured_pdf(path),
        "language": language,
        "size_kb": round(path.stat().st_size / 1024, 1),
        "pages": len(pages),
        "text_pages": len(pages) - len(ocr_pages),
        "pages_needing_ocr": len(ocr_pages),
        "ocr_page_numbers": ocr_pages[:10],
        "total_chars": sum(chars),
    })
    raw_pages.extend({"source": rel, "page": p["page"], "text": p["text"]} for p in pages)

inventory_df = pd.DataFrame(inventory)
if inventory_df.empty:
    raise RuntimeError(f"Every file failed to parse: {failed}")
inventory_df

,file,format,structured,language,size_kb,pages,text_pages,pages_needing_ocr,ocr_page_numbers,total_chars
0,1-2.pdf,pdf,True,en,174.9,23,23,0,[],38448
1,1-نشرة-بنجر-السكر_compressed.pdf,pdf,True,ar,231.4,26,25,1,[2],33412
2,135905761430.pdf,pdf,True,en,542.6,96,96,0,[],137717
3,2-البحث-الثاني.pdf,pdf,True,en,118.2,15,15,0,[],27683
4,2.pdf,pdf,True,en,165.6,21,21,0,[],32852
5,577137840-agriculture.pdf,pdf,True,en,100.7,14,14,0,[],22494
6,917873240-Key-Fact.pdf,pdf,True,en,3700.6,286,285,1,[2],346145
7,AGRI_008.pdf,pdf,True,ar,88.5,9,9,0,[],14352
8,AGRI_011_PDF.pdf,pdf,True,en,243.5,44,44,0,[],74538
9,AGRI_013.pdf,pdf,True,ar,643.7,83,83,0,[],160453


In [3]:
mostly_scanned = inventory_df[inventory_df.pages_needing_ocr > inventory_df.pages / 2]

print(f"Documents loaded   : {len(inventory_df)}  (failed to parse: {len(failed)}, "
      f"duplicate copies skipped: {len(duplicates)})")
print(f"Total pages        : {inventory_df.pages.sum()}")
print(f"Formats            : {inventory_df.format.value_counts().to_dict()}")
print(f"Structured (cleaned): {int(inventory_df.structured.sum())} of {len(inventory_df)} files")
print(f"Languages          : {inventory_df.language.value_counts().to_dict()}")
print(f"Pages needing OCR  : {inventory_df.pages_needing_ocr.sum()} "
      f"in {int((inventory_df.pages_needing_ocr > 0).sum())} file(s)")
print(f"Mostly-scanned docs: {mostly_scanned.file.tolist() or 'none'}")
print(f"Total characters   : {inventory_df.total_chars.sum():,}")
if failed:
    display(pd.DataFrame(failed))
if duplicates:
    display(pd.DataFrame(duplicates))

Documents loaded   : 52  (failed to parse: 0, duplicate copies skipped: 1)
Total pages        : 2841
Formats            : {'pdf': 52}
Structured (cleaned): 52 of 52 files
Languages          : {'ar': 40, 'en': 12}
Pages needing OCR  : 11 in 11 file(s)
Mostly-scanned docs: none
Total characters   : 4,149,545


,file,same_as
0,NEW_sugar_beet.pdf,1-نشرة-بنجر-السكر_compressed.pdf


In [4]:
# Look at real extracted text to spot what needs cleaning (headers, page numbers, broken words, tables)
for source in inventory_df.file.head(3):
    doc_pages = [p for p in raw_pages if p["source"] == source and len(p["text"].strip()) >= OCR_THRESHOLD]
    if not doc_pages:
        continue
    sample = doc_pages[len(doc_pages) // 2]
    print(f"===== {source} — page {sample['page']} (raw) =====")
    print(sample["text"][:700])
    print()

===== 1-2.pdf — page 12 (raw) =====
C 2 — I 1 — 2665.0 — 2520.0 — 63.45 — 60.00 — 57.47 — 54.15 — 90.57 — 90.25

C 2 — I 2 — 2600.0 — 2408.0 — 61.90 — 57.33 — 55.45 — 50.45 — 89.57 — 88.00

C 2 — I 3 — 2504.0 — 2330.0 — 59.62 — 55.48 — 54.00 — 48.90 — 90.56 — 88.14

C 2 — I 4 — 2431.0 — 2270.0 — 57.88 — 54.05 — 51.05 — 47.20 — 88.20 — 87.32

I 5 — 2388.0 — 2240.0 — 56.85 — 53.33 — 48.50 — 44.45 — 85.31 — 83.35

Mean C 2 — I 5 — 2517.6 — 2353.6 — 59.94 — 56.04 — 53.70 — 49.54 — 89.58 — 88.40

Mean C — I 5 — 2380.6 — 2216.3 — 56.68 — 52.77 — 49.80 — 45.88 — 87.86 — 87.05

Growing season 1 = 161 days, growing season 2 = 165 days, mean = 163 days

seasons with 15 days interval produced significant green fodder and seed yield.

I3,

===== 1-نشرة-بنجر-السكر_compressed.pdf — page 14 (raw) =====
## 25. أﻋﺮاض ﻧﻘﺺ اﻟﻤﻨﺠﻨﻴﺰ

ﺗﻈﻬﺮ أﻋﺮاض ﻧﻘﺺ اﻟﻤﻨﺠﻨﻴﺰ ﺑﻮﺟﻮد ﺑﻘﻊ ﺻﻔﺮاء ﺷﺎﺣﺒﺔ ا ﻟﻮن ﻋﻠﻲ ﺑﻌﺾ أﺟﺰاء ﻣﻦ اﻟﺄوراق ﺑﻴﻦ اﻟﻌﺮوق )اﻟﺄوﻋﻴﺔ( وﺗﺒﻘﻰ اﻟﻌﺮوق ﻓﻘﻂ ﺑﻠﻮن أﺧﻀﺮ ﺷﺎﺣﺐ.

ﻓﻰ ﺣﺎﻟﺔ اﻟﻨﻘﺺ اﻟﺸﺪﻳﺪ ﻳﺘﺤﻮل ﻟﻮن اﻟﺒﻘﻊ اﻟﺼﻔﺮ

### Findings ✏️ *(update with the numbers printed above after you run on your PDFs)*

- **How many documents / pages?** ___ PDF files, ___ pages in total (table above).
- **Formats:** all PDF (text-based). ___
- **Languages:** ___ Arabic, ___ English, ___ mixed (column `language`).
- **Duplicates:** ___ byte-identical copies skipped (table above); repeated passages inside and across
  files are dropped later, at chunk level (2.2).
- **Failed to parse / need OCR:** ___ files failed; ___ pages look scanned (listed in `ocr_page_numbers`).
  Scanned pages are skipped by the text pipeline — if a whole manual is scanned, run it through OCR
  first (e.g. `ocrmypdf input.pdf output.pdf`) and put the OCR'd copy in `data/raw/`.
- **Messy text noticed:** e.g. the same header/footer on every page, bare page numbers,
  words hyphenated across line breaks (`irriga-` / `tion`), fertilizer tables flattened into lines,
  ligatures such as `ﬁ`; in Arabic PDFs, letters extracted in their "presentation forms", diacritics,
  tatweel (ـ) and Persian letter variants. These are handled by the cleaning step below.

## 2.2 Chunking Strategy

### Cleaning first
Before chunking, each page is cleaned:
1. **Unicode normalisation** (NFKC) — turns ligatures like `ﬁ` into `fi`, odd spaces into normal ones,
   and Arabic *presentation forms* (the shaped glyphs PDFs store, e.g. `ﻢﻠﻌﻟا`) back into normal letters.
   **Undecodable glyphs** (`�`, ≈ 20 000 in six bulletins) are repaired: broken elongation marks are dropped
   (`الأراض��ى` → `الأراضى`) and letters a font draws in two pieces are restored to the real word
   (`م�سر` → `مصر`, `الن�سج` → `النضج`); this raises the share of real words in the zucchini and flax
   bulletins from ~86 % to ~94 %.
2. **Arabic clean-up** — diacritics (harakat) and tatweel (`ـ`) are removed so the same word always looks
   the same; Persian letter variants some bulletins use (`ی`, `ک`, `ھ`) become the Arabic letters
   (`ي`, `ك`, `ه`) so `ھذه` and `هذه` match; any page still extracted backwards is detected and flipped
   (Latin words and numbers inside it are kept the right way round).
3. **Repeated headers/footers removed** — a short line that appears on at least half of a document's
   pages (title bars, "Ministry of Agriculture", copyright lines) carries no meaning and would pollute
   every chunk. Digits are ignored when comparing, so "Page 3" and "Page 4" count as the same line.
4. **Bare page numbers removed** (`12`, `Page 12`, `صفحة ١٢`, `12 / 40`).
5. **Words re-joined across line breaks** (`fertili-\nzer` → `fertilizer`).
6. **Tables of contents removed** — a page with 5+ dot-leader lines (`مقدمة ........ 3`) keeps none of them
   (titles and page numbers match keyword searches but answer nothing); elsewhere the dots are removed.
7. **Whitespace normalised.**
7. **Unreadable pages skipped.** A page is dropped when fewer than 80 % of its words are real Arabic or
   English words (checked against the `wordfreq` word lists). This keeps the readable part of a partly
   garbled file (broken fonts, damaged scans) instead of indexing noise or dropping the whole file.

For the **cleaned PDFs** most of this has already happened when they were rebuilt (no covers, headers,
page numbers, contents pages or broken glyphs are left); the same steps still run as a safety net, and the
`#` heading lines pass through untouched so the chunker can use them.

A font in `AGRI_021` encodes the stretching stroke as the letter `ر` (`الزراعيرة`, `المررجح`); a word that is
not a real word but becomes a common one without one `ر` is repaired. Cleaned PDFs use a gentler
unreadable-page threshold (60 % real words) because pages of scientific names are real content.


In [5]:
PERSIAN_TO_ARABIC = str.maketrans({"ی": "ي", "ک": "ك", "ھ": "ه", "ۀ": "ه", "ہ": "ه"})
PAGE_NUMBER_RE = re.compile(r"^\s*(page|صفحة|ص\.?)?\s*\d{1,4}(\s*(of|/|من)\s*\d{1,4})?\s*$", re.IGNORECASE)

# Frequent Arabic words: if mostly their mirror images appear, the page was extracted backwards
_AR_COMMON = {"في", "من", "على", "إلى", "الى", "عن", "أن", "التي", "الذي", "هذا", "هذه", "مع", "أو", "ذلك", "كما"}
_AR_COMMON_REVERSED = {w[::-1] for w in _AR_COMMON}
_ARABIC_WORD_RE = re.compile(r"[\u0621-\u064A]+")


def looks_reversed(text: str) -> bool:
    words = _ARABIC_WORD_RE.findall(text)
    normal = sum(w in _AR_COMMON for w in words) + sum(w.startswith("ال") and len(w) > 3 for w in words)
    backwards = sum(w in _AR_COMMON_REVERSED for w in words) + sum(w.endswith("لا") and len(w) > 3 for w in words)
    return backwards >= 5 and backwards > 2 * normal


def unreverse_line(line: str) -> str:
    """Flip a visually-ordered RTL line back to reading order, keeping LTR runs (numbers, Latin) intact."""
    if not _ARABIC_WORD_RE.search(line):
        return line
    return _LTR_RUNS_RE.sub(lambda m: m.group(0)[::-1], line[::-1])


def normalise_line(line: str) -> str:
    return re.sub(r"\d+", "#", line.strip().lower())


BROKEN_WORD_RE = re.compile(r"[\u0600-\u06ff\ufffd]*\ufffd[\u0600-\u06ff\ufffd]*")


def repair_word(match: re.Match) -> str:
    """An Arabic word with undecodable glyphs ("�", U+FFFD) -> the most likely real word.

    Some bulletins' fonts can't be decoded: elongation (kashida) glyphs come out as "��"
    (الأراض��ى), and ص / ض / ش / س drawn as two pieces come out as "�" + س or ص (م�سر = مصر,
    الن�سج = النضج, للإر�ساد = للإرشاد). The elongation marks are dropped; for the letters, every
    candidate is tried and the one that makes the most common real word (wordfreq) is kept.
    """
    word = re.sub(r"\ufffd{2,}", "", match.group(0))
    pieces = re.split(r"\ufffd[سصضش]", word)
    if 1 < len(pieces) <= 4:
        options = ["".join(p + l for p, l in zip(pieces, combo + ("",))).replace("\ufffd", "")
                   for combo in itertools.product("سصضش", repeat=len(pieces) - 1)]
        best = max(options, key=lambda o: zipf_frequency(o, "ar"))
        if zipf_frequency(best, "ar") > 0:
            return best
    return word.replace("\ufffd", "")


def repair_broken_glyphs(text: str) -> str:
    return BROKEN_WORD_RE.sub(repair_word, text) if "\ufffd" in text else text


_STRAY_RA_WORD_RE = re.compile(r"[\u0621-\u064A]*ر[\u0621-\u064A]*")
_stray_cache: dict[str, str] = {}


def repair_stray_ra(text: str) -> str:
    """Some fonts encode the stretching stroke (kashida) as the letter ر, so justified words gain one:
    الزراعيرة, المررجح, تتزايرد. A word that is not a real word but becomes a common one without one of its
    ر is repaired (الزراعية, المرجح, تتزايد); real words are never touched."""
    def fix(m: re.Match) -> str:
        word = m.group(0)
        if word not in _stray_cache:
            best = word
            if len(word) >= 4 and zipf_frequency(word, "ar") == 0:
                options = {word[:i] + word[i + 1:] for i, ch in enumerate(word) if ch == "ر"}
                good = max(options, key=lambda o: zipf_frequency(o, "ar"), default=word)
                if zipf_frequency(good, "ar") >= 2.0:
                    best = good
            _stray_cache[word] = best
        return _stray_cache[word]
    return _STRAY_RA_WORD_RE.sub(fix, text)


_SWAP_WORD_RE = re.compile(r"[\u0621-\u064A]{3,}")


def swapped_letter_fixes(texts: list[str]) -> dict[str, str]:
    """Word repairs for one document whose font wrote some letters with the wrong glyph.
    - 'اإستخدام', 'اأو' (an alef in front of إ / أ) -> 'إستخدام', 'أو': never a real word, fixed everywhere.
    - one bulletin draws ص, ش and ض with the glyph of س (المحسول, انتسار, اخسر). That is only repaired
      when it happens 20+ times in the document, so a rare real word elsewhere is never 'corrected'."""
    alef, swaps, words = {}, {}, Counter(w for t in texts for w in _SWAP_WORD_RE.findall(t))
    for w, n in words.items():
        if zipf_frequency(w, "ar") > 0:
            continue
        if w[:2] in ("اإ", "اأ") and zipf_frequency(w[1:], "ar") >= 2.0:
            alef[w] = w[1:]
        elif "س" in w:
            options = [w[:i] + new + w[i + 1:] for i, ch in enumerate(w) if ch == "س" for new in "صشض"]
            best = max(options, key=lambda o: zipf_frequency(o, "ar"))
            if zipf_frequency(best, "ar") >= 2.5:
                swaps[w] = best
    if sum(words[w] for w in swaps) < 20:
        swaps = {}
    return {**alef, **swaps}


def apply_word_fixes(text: str, fixes: dict[str, str]) -> str:
    return _SWAP_WORD_RE.sub(lambda m: fixes.get(m.group(0), m.group(0)), text) if fixes else text


def prepare_page(text: str) -> tuple[str, bool]:
    """Unicode + Arabic normalisation, done before anything compares lines.
    Returns (text, whether the page had to be un-reversed)."""
    text = unicodedata.normalize("NFKC", text).replace("\u00ad", "")  # ligatures, Arabic forms, soft hyphens
    text = re.sub(r"\(cid:\d+\)\s?", "", text)                          # glyphs with no Unicode (bullets, ticks)
    text = repair_broken_glyphs(text)                                 # الأراض��ى, م�سر -> الأراضى, مصر
    text = repair_stray_ra(text)                                      # الزراعيرة -> الزراعية
    text = strip_arabic_marks(text)                                   # harakat + tatweel
    text = text.translate(PERSIAN_TO_ARABIC)                          # ی ک ھ -> ي ك ه
    reversed_page = looks_reversed(text)
    if reversed_page:
        text = "\n".join(unreverse_line(l) for l in text.splitlines())
    return text, reversed_page


def find_repeated_lines(page_texts: list[str], min_share: float = 0.5) -> set[str]:
    """Short lines that appear on at least `min_share` of a document's pages (headers/footers)."""
    if len(page_texts) < 4:
        return set()
    counts = Counter()
    for text in page_texts:
        # heading lines ("## التسميد") of the cleaned PDFs are structure, never page furniture
        counts.update({normalise_line(l) for l in text.splitlines()
                       if 0 < len(l.strip()) <= 80 and not l.startswith("#")})
    return {line for line, n in counts.items() if n / len(page_texts) >= min_share}


LEADER_RE = re.compile(r"(?:[ \t]*[.…_][ \t]?){5,}[ \t]*\d{0,4}")  # "..........  27" (contents dots + page)


def clean_page(text: str, repeated: set[str]) -> str:
    """`text` has already been through prepare_page()."""
    text = re.sub(r"(\w)-\n(?=[a-z])", r"\1", text)                   # irriga-\ntion -> irrigation
    lines = text.splitlines()
    # A table of contents (5+ lines with dot leaders) is only titles and page numbers: drop those
    # lines, or every section title would match keyword searches. Elsewhere just remove the dots.
    toc = sum(bool(LEADER_RE.search(l)) for l in lines) >= 5
    kept = []
    for line in lines:
        if PAGE_NUMBER_RE.match(line) or normalise_line(line) in repeated or (toc and LEADER_RE.search(line)):
            continue
        line = LEADER_RE.sub(" ", line)
        kept.append(re.sub(r"[ \t]+", " ", line).strip())
    return re.sub(r"\n{3,}", "\n\n", "\n".join(kept)).strip()


MIN_PAGE_VALID_WORDS = 0.80   # pages with less than this share of real words are skipped
# Cleaned PDFs hold real text already; only a page that is mostly broken font is skipped (a page of
# scientific names, e.g. Cuscuta / Isoproturon, scores 0.7 and must stay)
MIN_PAGE_VALID_WORDS_CLEAN = 0.60
_AR_WORD_RE, _EN_WORD_RE = re.compile("[\u0621-\u064A]{2,}"), re.compile("[A-Za-z]{3,}")
_word_cache = {}


def valid_word_share(text: str) -> tuple[float, int]:
    words = [(w, "ar") for w in _AR_WORD_RE.findall(text)] + [(w.lower(), "en") for w in _EN_WORD_RE.findall(text)]
    for w in words:
        if w not in _word_cache:
            _word_cache[w] = zipf_frequency(*w) > 0
    return (sum(_word_cache[w] for w in words) / len(words) if words else 1.0), len(words)


clean_pages = []
removed_headers = {}
reversed_pages = Counter()
unreadable_pages = Counter()
letter_fixes = Counter()
structured_files = set(inventory_df.loc[inventory_df.structured, "file"])
for source in inventory_df.file:
    doc = [p for p in raw_pages if p["source"] == source]
    min_valid = MIN_PAGE_VALID_WORDS_CLEAN if source in structured_files else MIN_PAGE_VALID_WORDS
    prepared = [prepare_page(p["text"]) for p in doc]
    fixes = swapped_letter_fixes([text for text, _ in prepared])
    if fixes:
        letter_fixes[source] = len(fixes)
        prepared = [(apply_word_fixes(text, fixes), was_reversed) for text, was_reversed in prepared]
    repeated = find_repeated_lines([text for text, _ in prepared])
    removed_headers[source] = sorted(repeated)
    for p, (prepared_text, was_reversed) in zip(doc, prepared):
        text = clean_page(prepared_text, repeated)
        reversed_pages[source] += was_reversed
        if len(text) < OCR_THRESHOLD:  # blank / scanned page
            continue
        share, n_words = valid_word_share(text)
        if n_words >= 40 and share < min_valid:  # garbled page
            unreadable_pages[source] += 1
            continue
        clean_pages.append({"source": source, "page": p["page"], "text": text,
                            "lang": detect_language(text)})

print(f"Pages with usable text after cleaning: {len(clean_pages)} / {len(raw_pages)}")
print(f"Words repaired (wrong-glyph letters): {dict(letter_fixes) or 'none'}")
print(f"Page languages: {dict(Counter(p['lang'] for p in clean_pages))}")
print(f"Unreadable pages skipped: {sum(unreadable_pages.values())} {dict(unreadable_pages) or ''}")
print(f"Arabic pages fixed for reversed extraction: {sum(reversed_pages.values())} "
      f"{ {k: v for k, v in reversed_pages.items() if v} or ''}")
for source, lines in removed_headers.items():
    if lines:
        print(f"  {source}: removed repeated lines {lines[:4]}")

Pages with usable text after cleaning: 2826 / 2841
Words repaired (wrong-glyph letters): {'الكانتلوب.pdf': 6, 'الكتان2020_compressed.pdf': 5, 'نشرة-الكوسة.pdf': 171}
Page languages: {'en': 1134, 'ar': 1692}
Unreadable pages skipped: 4 {'917873240-Key-Fact.pdf': 1, 'cd7067en.pdf': 1, 'res4_-mohamed-abdel-hady-sayed-abdel-wahab.pdf': 2}
Arabic pages fixed for reversed extraction: 0 
  الكتان2020_compressed.pdf: removed repeated lines ['الإدارة المركزية للإرشاد الزراعي']


In [6]:
# Before / after on one page per language
for lang in ("ar", "en"):
    page = next((c for c in clean_pages if c["lang"] == lang and len(c["text"]) > 150), None)
    if page is None:
        continue
    raw = next(p["text"] for p in raw_pages if p["source"] == page["source"] and p["page"] == page["page"])
    print(f"===== {lang.upper()} · {page['source']} p.{page['page']} =====")
    print("BEFORE:\n", raw[:400], "\n")
    print("AFTER:\n", page["text"][:400], "\n")

===== AR · 1-نشرة-بنجر-السكر_compressed.pdf p.1 =====
BEFORE:
 # زراﻋﺔ وإﻧﺘﺎج ﺑﻨﺠﺮ اﻟﺴﻜﺮ

ﻳﺘﻨﺎول ﻫﺬا اﻟﻤﺴﺘﻨﺪ: ﻣﻘﺪﻣﺔ، إﻋﺪاد اﻟﺄرض ﻟﻠﺰراﻋﺔ، ﻛﻤﻴﺔ وﻧﻮع اﻟﺘﻘﺎوى، ﻣﻴﻌﺎد اﻟﺰراﻋﺔ، ﻃﺮﻳﻘﺔ اﻟﺰراﻋﺔ، رى اﻟﺒﻨﺠﺮ، ﻋﻤﻠﻴﺔ اﻟﺘﺮﻗﻴﻊ، ﻋﻤﻠﻴﺔ اﻟﺨﻒ، اﻟﻌﺰﻳﻖ وﻣﻜﺎﻓﺤﺔ اﻟﺤﺸﺎﺋﺶ، اﻟﺘﺴﻤﻴﺪ، أﻋﺮاض ﻧﻘﺺ اﻟﻨﻴﺘﺮوﺟﻴﻦ، ﺗﺄﺛﻴﺮ اﻟﻨﻴﺘﺮوﺟﻴﻦ ﻋﻠﻰ اﻟﺴﻜﺮوز، ﺗﺄﺛﻴﺮ اﻟﻨﻴﺘﺮوﺟﻴﻦ ﻋﻠﻰ اﻟﺠﻮدة، ﺗﺄﺛﻴﺮ اﻟﻨﻴﺘﺮوﺟﻴﻦ ﻋﻠﻰ ﻣﺤﺼﻮل اﻟﺠﺬور، ﻛﻤﻴﺔ وﻣﻴﻌﺎد وﺻﻮرة إﺿﺎﻓﺔ اﻟﻨﻴﺘﺮوﺟﻴﻦ … 

AFTER:
 # زراعة وإنتاج بنجر السكر

يتناول هذا المستند: مقدمة، إعداد الأرض للزراعة، كمية ونوع التقاوى، ميعاد الزراعة، طريقة الزراعة، رى البنجر، عملية الترقيع، عملية الخف، العزيق ومكافحة الحشائش، التسميد، أعراض نقص النيتروجين، تأثير النيتروجين على السكروز، تأثير النيتروجين على الجودة، تأثير النيتروجين على محصول الجذور، كمية وميعاد وصورة إضافة النيتروجين ... 

===== EN · 1-2.pdf p.1 =====
BEFORE:
 # Water Productivity for Egyptian Clover as Affected by Different Irrigation Regimes and Cultivation Methods in The North Middle Nile Delta Region A2F I0

This document covers: Introduction

### Chosen strategy: section-aware chunks with heading context (cleaned PDFs), sentence-aware fixed-size chunks otherwise

**Cleaned PDFs — one section per chunk, with its heading path.** In the rebuilt PDFs the headings are
reliable, so a chunk never crosses from one `##` section into the next (a fertiliser chunk is never half
irrigation), and every chunk starts with its heading path, e.g.
`زراعة وانتاج زهرة الشمس › التسميد › ثانيا: التسميد في الأراضي الجديدة`.
A paragraph in the middle of a bulletin rarely repeats the crop or the topic ("يضاف 45 كجم آزوت للفدان …"),
so without that line its embedding and its keywords don't say *sunflower* or *fertiliser*; with it,
both searches match questions that name the crop or the practice, and the LLM sees which section a
passage comes from. Inside a section the same size, overlap and sentence rules as below apply
(the header counts towards the 800 characters); sub-section headings stay in the text as their own
lines, and a long section is split into several chunks that each repeat the header.

### Fixed-size rules (all documents)

**Chunk size = 800 characters (≈170 tokens in English, ≈200–250 in Arabic).**
Farming guides give advice in short, self-contained passages: a planting window, a fertilizer rate,
the symptoms of a pest and how to treat it. About 800 characters is enough to hold one such
recommendation *together with its conditions* (which crop, which growth stage, which units), without
mixing in a second topic that would blur the embedding. It also stays well under the **512-token limit of
`multilingual-e5-base`** — Arabic needs more tokens per character than English, and anything over the
limit is silently truncated, so much bigger chunks would be partly invisible to the retriever (verified in 2.3).

**Overlap = 150 characters, in whole sentences.**
A recommendation such as *"apply 50 kg/ha of urea … when the crop is knee-high"* often straddles a chunk
boundary. Repeating the last sentence(s) of one chunk at the start of the next keeps it intact in at
least one chunk. Overlap is capped at half a chunk so the store doesn't hold everything twice.

**Never cut mid-sentence or mid-number.** Text is split on sentence ends (`.` `!` `?` and the Arabic
question mark `؟`) and list items, then packed
into chunks. This keeps doses and units together (`120 kg N/ha` is never split), which matters because
farmers often ask for exact quantities. Only a single sentence longer than a whole chunk is cut by words.

**Chunks can cross page breaks**, and each chunk records `page_start`/`page_end` for the citation.

**Repeated passages are dropped.** Some files repeat their own paragraphs, and some bulletins exist in
two editions that share most of their text. Copies would fill the top-k results with the same passage,
so two passes remove them while keeping the first copy and anything new in the later file:
1. before chunking, a sentence of 40+ characters that already appeared (in any earlier file or page) is skipped;
2. after chunking, a chunk whose 8-word sequences are ≥ 80 % already indexed is dropped.

*Alternatives considered:* section-based chunking was not possible on the original PDFs (their headings
were not reliably extractable); the cleaned corpus made it possible, so it is used there. Smaller/larger
fixed sizes are compared empirically in **2.6b**.

In [7]:
BULLET_RE = re.compile(r"^([•▪●◦\-–*]|\d{1,2}[.)\-]|[a-zأ-ي][.)\-])\s+")
# Split after . ! ? ؟ when the next word does not start with a lower-case Latin letter
# (Arabic has no capitals, so "next char is uppercase" would never split Arabic text).
SENTENCE_RE = re.compile(r"(?<=[.!?؟])\s+(?=[^\sa-z])")


def split_into_units(text: str) -> list[str]:
    """Paragraphs -> units. Wrapped lines are joined; headings and list items stay separate."""
    units = []
    for paragraph in re.split(r"\n\s*\n", text):
        buffer = ""
        for line in (l.strip() for l in paragraph.split("\n")):
            if not line:
                continue
            last_line_short = bool(buffer) and len(buffer.rsplit("\n", 1)[-1]) < 60
            if buffer and (BULLET_RE.match(line) or last_line_short):
                units.append(buffer)
                buffer = line
            else:
                buffer = f"{buffer} {line}".strip()
        if buffer:
            units.append(buffer)
    return units


def split_sentences(unit: str) -> list[str]:
    return [s.strip() for s in SENTENCE_RE.split(unit) if s.strip()]


def hard_split(sentence: str, size: int) -> list[str]:
    """Only for a single 'sentence' longer than a whole chunk (e.g. a flattened table)."""
    pieces, current = [], ""
    for word in sentence.split():
        if current and len(current) + 1 + len(word) > size:
            pieces.append(current)
            current = word
        else:
            current = f"{current} {word}".strip()
    return pieces + ([current] if current else [])


def document_sentences(pages: list[dict], size: int) -> list[dict]:
    out = []
    for p in pages:
        for unit in split_into_units(p["text"]):
            for i, sentence in enumerate(split_sentences(unit)):
                for j, piece in enumerate(hard_split(sentence, size) if len(sentence) > size else [sentence]):
                    out.append({"text": piece, "page": p["page"], "new_unit": i == 0 and j == 0})
    return out


def pack(sentences: list[dict], size: int, overlap: int) -> list[list[dict]]:
    chunks, current, length = [], [], 0
    for s in sentences:
        n = len(s["text"]) + 1
        if current and length + n > size:
            chunks.append(current)
            # carry over whole trailing sentences: at least `overlap` chars, at most half a chunk
            keep, kept = [], 0
            for prev in reversed(current):
                if kept >= overlap or kept + len(prev["text"]) + 1 > size // 2:
                    break
                keep.insert(0, prev)
                kept += len(prev["text"]) + 1
            current, length = keep, kept
            while current and length + n > size:  # make room for the new sentence
                length -= len(current.pop(0)["text"]) + 1
        current.append(s)
        length += n
    if current:
        chunks.append(current)
    return chunks


def slug(text: str) -> str:
    """Readable, unique id prefix; keeps Arabic letters and adds a hash so ids never collide."""
    import hashlib
    readable = re.sub(r"[^\w]+", "_", Path(text).stem).strip("_").lower()[:40]
    return f"{readable}_{hashlib.md5(text.encode('utf-8')).hexdigest()[:6]}"


repeated_sentences = Counter()  # filled by chunk_corpus: sentences skipped because they appeared before


# ---------------------------------------------------------------- section-aware chunks (cleaned PDFs)
HEADING_RE = re.compile(r"^(#{1,4}) (.+)$")


def clean_heading(text: str) -> str:
    """'10. 5. التسميد :-' -> 'التسميد' (numbering and trailing colons are layout, not meaning)."""
    text = re.sub(r"\(\d+:[a-z]+\)\s*", "", text)   # '(0:dic)' left by some source PDFs
    for _ in range(2):
        text = re.sub(r"^\s*\d{1,3}\s*[.\-)]\s*", "", text)
    return text.strip(" :-–").strip()


def is_structured_doc(pages: list[dict]) -> bool:
    return any(HEADING_RE.match(line) for p in pages for line in p["text"].splitlines())


def structured_sentences(pages: list[dict], size: int) -> tuple[str, list[dict]]:
    """(document title, sentences) where every sentence knows its section path.
    A section is the document's highest heading level below the title (## in most documents)."""
    levels = [len(m.group(1)) for p in pages for line in p["text"].splitlines()
              if (m := HEADING_RE.match(line)) and len(m.group(1)) > 1]
    top = min(levels, default=2)
    title, path, section_no, out = "", {}, 0, []
    for p in pages:
        for block in re.split(r"\n\s*\n", p["text"]):
            block = " ".join(l.strip() for l in block.splitlines() if l.strip())
            if not block:
                continue
            m = HEADING_RE.match(block)
            if m:
                level, text = len(m.group(1)), clean_heading(m.group(2))
                if level == 1:
                    title = title or text
                    continue
                path = {k: v for k, v in path.items() if k < level}
                path[level] = text
                section_no += level == top
                # the heading stays in the text as its own line: it names what the next lines are about
                out.append({"text": text, "page": p["page"], "new_unit": True,
                            "section": (section_no, path.get(top, "")), "path": dict(path)})
                continue
            for i, sentence in enumerate(split_sentences(block)):
                pieces = hard_split(sentence, size) if len(sentence) > size else [sentence]
                for j, piece in enumerate(pieces):
                    out.append({"text": piece, "page": p["page"], "new_unit": i == 0 and j == 0,
                                "section": (section_no, path.get(top, "")), "path": dict(path)})
    return title, out


def chunk_header(title: str, path: dict) -> str:
    """'زراعة وانتاج زهرة الشمس › التسميد › ثانيا: التسميد في الأراضي الجديدة'"""
    parts = [title] + [path[k] for k in sorted(path)]
    seen, kept = set(), []
    for part in parts:
        if part and part not in seen:
            seen.add(part)
            kept.append(part)
    return " › ".join(kept)


# Readable titles for files whose cleaned PDF kept a file-name title (the title starts every chunk
# header, so it is also searched). Add a line here when you add a file with a meaningless name.
TITLE_OVERRIDES = {
    "577137840-agriculture.pdf": "General Knowledge: Agriculture (Agronomy, Seed Science, Agroforestry, Animal Husbandry)",
    "AGRI_021.pdf": "الأمن الغذائي وتغير المناخ",
    "AGRI_013.pdf": "مدونة السلوك الدولية بشأن توزيع المبيدات واستعمالها",
    "AGRI_008.pdf": "الحفظ والتوسع: تكثيف إنتاج المحاصيل المستدام",
    "Unit 1.pdf": "Environment and Agriculture: Environmental Concerns",
    "res4_-mohamed-abdel-hady-sayed-abdel-wahab.pdf": "Response of Wheat to Ammonium and Nitrate Nitrogen on Sandy Soil",
    "135905761430.pdf": "Agricultural Knowledge and Information Systems and Poverty Reduction",
    "1-2.pdf": "Water Productivity for Egyptian Clover under Different Irrigation Regimes and Cultivation Methods",
    "2.pdf": "Effect of Magnetic Irrigation Water and Nitrogen Fertilizer Forms on Maize",
    "2-البحث-الثاني.pdf": "Organic Wastes as Growing Media for Date Palm Plantlets during Acclimatization",
    "farmerbook.pdf": "Farmer’s Handbook on Basic Agriculture",
    "ticket_file6214bd8f64479-Pomme-de-terre-Ardna (1).pdf": "دليل الفلاح: زراعة البطاطس",
    "NEW_sugarcane_seedlings.pdf": "إنتاج شتلات قصب السكر",
    "الكتان2020_compressed.pdf": "زراعة وإنتاج الكتان",
    "نخيل-البلح-نشر.pdf": "نخيل البلح",
    "شجرة-الزيتون-نشرة-10.pdf": "شجرة الزيتون",
    "نشرة-المانجو-طبع_compressed.pdf": "نشرة المانجو",
    "نشرة-فول-الصويا-طبع_compressed.pdf": "نشرة فول الصويا",
    "نشرة-البصل-2022.pdf": "نشرة البصل",
    "نشرة-الذرة-نهائى-copy_compressed.pdf": "زراعة وإنتاج الذرة الشامية",
}


def chunk_corpus(pages: list[dict], size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[dict]:
    chunks, seen = [], set()
    repeated_sentences.clear()
    for source in dict.fromkeys(p["source"] for p in pages):  # keeps file order
        doc_pages = [p for p in pages if p["source"] == source]
        structured = is_structured_doc(doc_pages)
        title, doc_sentences = structured_sentences(doc_pages, size) if structured \
            else ("", document_sentences(doc_pages, size))
        title = TITLE_OVERRIDES.get(source, title)
        sentences = []
        for s in doc_sentences:
            key = re.sub(r"\W+", " ", normalize_for_match(s["text"])).strip()
            if len(key) >= 40:  # a real sentence, not a heading like "التسميد:"
                if key in seen:  # already indexed earlier (same file or another file)
                    repeated_sentences[source] += 1
                    continue
                seen.add(key)
            sentences.append(s)
        # cleaned PDFs: pack each ## section on its own; other documents: one stream of sentences
        groups = [list(g) for _, g in itertools.groupby(sentences, key=lambda s: s.get("section"))] \
            if structured else [sentences]
        idx = 0
        for group in groups:
            header = chunk_header(title, group[0]["path"]) if structured else ""
            budget = max(size - len(header) - 1, size // 2)
            for packed in pack(group, budget, overlap):
                if structured:
                    header = chunk_header(title, packed[0]["path"])
                    if packed[0]["text"] == header.rsplit(" › ", 1)[-1]:
                        packed = packed[1:]  # the section heading is already the end of the header
                    if not packed:
                        continue
                body = ""
                for k, s in enumerate(packed):
                    sep = "" if k == 0 else ("\n" if s["new_unit"] else " ")
                    body += sep + s["text"]
                if len(body) < MIN_CHUNK_CHARS:
                    continue
                text = f"{header}\n{body}" if header else body
                chunks.append({
                    "id": f"{slug(source)}-{idx:04d}",
                    "text": text,
                    "body": body,  # without the header, for the duplicate check
                    "metadata": {
                        "source": source,
                        "page_start": min(s["page"] for s in packed),
                        "page_end": max(s["page"] for s in packed),
                        "chunk_index": idx,
                        "n_chars": len(text),
                        "lang": detect_language(body),
                        "title": title,
                        "section": packed[0]["section"][1] if structured else "",
                    },
                })
                idx += 1
    return chunks


def _shingles(text: str, n: int = 8) -> set[int]:
    words = re.findall(r"\w+", normalize_for_match(text))
    return {hash(" ".join(words[i:i + n])) for i in range(max(len(words) - n + 1, 1))}


def drop_duplicate_chunks(chunks: list[dict], max_seen: float = 0.8) -> tuple[list[dict], Counter]:
    """Drop chunks whose text (8-word sequences) mostly appeared in an earlier chunk."""
    seen, kept, dropped = set(), [], Counter()
    for c in chunks:
        sh = _shingles(c.get("body", c["text"]))
        if len(sh & seen) / len(sh) >= max_seen:
            dropped[c["metadata"]["source"]] += 1
            continue
        seen |= sh
        kept.append(c)
    return kept, dropped


all_chunks = chunk_corpus(clean_pages)
chunks, dropped_chunks = drop_duplicate_chunks(all_chunks)
chunk_df = pd.DataFrame([{**c["metadata"], "id": c["id"]} for c in chunks])
print(f"{len(chunks)} chunks from {chunk_df.source.nunique()} documents, "
      f"{int((chunk_df.section != '').sum())} of them section-aware with a heading path "
      f"({sum(repeated_sentences.values())} repeated sentences and {sum(dropped_chunks.values())} repeated chunks dropped)")
if repeated_sentences or dropped_chunks:
    display(pd.DataFrame({"repeated sentences skipped": pd.Series(repeated_sentences),
                          "repeated chunks dropped": pd.Series(dropped_chunks)}).fillna(0).astype(int)
            .sort_values("repeated sentences skipped", ascending=False).head(15))
display(chunk_df.n_chars.describe().round(0).to_frame("chunk length (chars)").T)
display(chunk_df.groupby(["source", "lang"]).size().rename("chunks").to_frame())

8872 chunks from 52 documents, 8491 of them section-aware with a heading path (2167 repeated sentences and 145 repeated chunks dropped)


,repeated sentences skipped,repeated chunks dropped
Agricultural_RAG_200_Pages_Arabic.pdf,1261,33
cd7067en.pdf,200,3
القمح-قديم-2022_compressed.pdf,154,14
القمح-فى-الأراضى-الجديدة_compressed.pdf,138,37
نشرة-الذرة-نهائى-copy_compressed.pdf,86,15
farmerbook.pdf,64,1
نشرة-االقصب-2023-copy_compressed.pdf,56,22
الموالح_compressed.pdf,51,1
نشرة-البصل-2022.pdf,38,2
135905761430.pdf,22,5


,count,mean,std,min,25%,50%,75%,max
chunk length (chars),8872.0,629.0,206.0,81.0,537.0,708.0,779.0,1011.0


,,chunks
source,lang,
1-2.pdf,en,88
1-نشرة-بنجر-السكر_compressed.pdf,ar,77
135905761430.pdf,en,311
2-البحث-الثاني.pdf,en,61
2.pdf,en,72
577137840-agriculture.pdf,en,50
917873240-Key-Fact.pdf,en,674
AGRI_008.pdf,ar,32
AGRI_011_PDF.pdf,en,169


In [8]:
# Two neighbouring chunks per language: note the overlapping sentence(s) at the boundary
neighbours = []
for lang in ("ar", "en"):
    lang_chunks = [c for c in chunks if c["metadata"]["lang"] == lang]
    if lang_chunks:
        doc = lang_chunks[0]["metadata"]["source"]
        neighbours += [c for c in lang_chunks if c["metadata"]["source"] == doc][:2]
for c in neighbours:
    m = c["metadata"]
    print(f"--- {c['id']}  ({m['source']}, pages {m['page_start']}-{m['page_end']}, {m['n_chars']} chars, "
          f"section: {m.get('section') or '-'}) ---")
    print(c["text"], "\n")

--- 1_نشرة_بنجر_السكر_compressed_307e05-0000  (1-نشرة-بنجر-السكر_compressed.pdf, pages 1-1, 346 chars, section: -) ---
زراعة وإنتاج بنجر السكر
يتناول هذا المستند: مقدمة، إعداد الأرض للزراعة، كمية ونوع التقاوى، ميعاد الزراعة، طريقة الزراعة، رى البنجر، عملية الترقيع، عملية الخف، العزيق ومكافحة الحشائش، التسميد، أعراض نقص النيتروجين، تأثير النيتروجين على السكروز، تأثير النيتروجين على الجودة، تأثير النيتروجين على محصول الجذور، كمية وميعاد وصورة إضافة النيتروجين ... 

--- 1_نشرة_بنجر_السكر_compressed_307e05-0001  (1-نشرة-بنجر-السكر_compressed.pdf, pages 3-3, 733 chars, section: مقدمة) ---
زراعة وإنتاج بنجر السكر › مقدمة
يمكن زراعة بنجر السكر فى جميع الأراضى – وهو من المحاصيل الحقلية المتحملة للملوحة والقلوية بدرجة كبيرة بمجرد تخطيه مرحلة البادرة دون أن تجف البادرات وتموت - لذا فهو من المحاصيل المصلحة للتربة حيث يختزن الأملاح – إلا أن أفضل الأراضى لزراعة البنجر هى الأراضى الصفراء – ومن المهم جدا أن تكون الأرض جيدة الصرف.
يعتبر أقصى عدد من نباتات بنجر السكر يمكن زراعتها بالفدان والحفاظ عليها 

### What is each document about? (topic labels)
A chunk in the middle of a bulletin rarely repeats the crop's name: the seed-rate paragraph of the
sunflower bulletin says `معدل التقاوي ... 4 – 5 كجم بذرة / فدان` without "sunflower", and the palm
bulletin's planting-date paragraph talks about `الفسيلة` without `النخيل`. A question naming the crop then
can't match it by keywords, and the LLM can't tell which crop the passage is about.

So each document gets a **topic**: the crop named in its file name, or else the crop it keeps mentioning
(in ≥ 25 % of its chunks and more often than all other crops together), using the crop list of the
glossary (`backend/app/services/glossary.py`). General documents (FAO reports, handbooks covering many
crops) get none. The topic is stored in each chunk's metadata; the keyword search indexes it with the
chunk, and the prompt shows it next to the source (`about: دوار الشمس / sunflower`). The embedded text is
unchanged.

In [9]:
from app.services.glossary import document_topic

doc_chunks = {}
for c in chunks:
    doc_chunks.setdefault(c["metadata"]["source"], []).append(c["text"])
topics = {}
for source, texts in doc_chunks.items():
    term = document_topic(texts, source)
    topics[source] = f"{term.ar} / {term.en}" if term else ""
for c in chunks:
    c["metadata"]["topic"] = topics[c["metadata"]["source"]]

print(f"{sum(bool(t) for t in topics.values())} of {len(topics)} documents have a topic")
pd.DataFrame(sorted(topics.items()), columns=["document", "topic"]).replace("", "— (general)")

39 of 52 documents have a topic


,document,topic
0,1-2.pdf,البرسيم / clover
1,1-نشرة-بنجر-السكر_compressed.pdf,بنجر السكر / sugar beet
2,135905761430.pdf,— (general)
3,2-البحث-الثاني.pdf,نخيل البلح / date palm
4,2.pdf,الذرة الشامية / maize
5,577137840-agriculture.pdf,— (general)
6,917873240-Key-Fact.pdf,— (general)
7,AGRI_008.pdf,— (general)
8,AGRI_011_PDF.pdf,— (general)
9,AGRI_013.pdf,— (general)


## 2.3 Embeddings & Vector Store

- **Embedding model:** `BAAI/bge-m3` — multilingual (100+ languages, strong on Arabic), 1024-dimensional,
  8192-token window, ≈2.2 GB (a GPU makes embedding the corpus take minutes instead of an hour).
  Arabic and English text land in the **same vector space**, so a question in Arabic retrieves English
  passages that answer it (and the other way round). bge-m3 needs no prefixes; for e5 models the
  `query: ` / `passage: ` prefixes are set automatically and saved in `rag_config.json`.
- **Egyptian Arabic:** when `models/bge-m3-egyptian/` exists (made by `notebooks/egyptian_finetune.ipynb`), the
  same bge-m3 fine-tuned on Egyptian farming questions is used instead — same size and vector space, trained only
  in its top layers on questions generated from these documents.
  Embeddings are L2-normalised so cosine similarity is a simple dot product.
- *Alternatives:* `intfloat/multilingual-e5-base` (768-d, ≈1.1 GB) runs comfortably on a CPU-only laptop
  with somewhat lower quality on Arabic; `intfloat/multilingual-e5-small` is faster still.
- **Vector store:** Chroma, persisted to `backend/data/vector_store/` with cosine distance.
  The backend opens this folder at startup — nothing is rebuilt per request.

In [10]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(model_location(EMBEDDING_MODEL))
EMBEDDING_DIM = int(embedder.encode(["test"]).shape[1])
MAX_TOKENS = embedder.max_seq_length
print(f"Model: {EMBEDDING_MODEL} | dimension: {EMBEDDING_DIM} | max tokens: {MAX_TOKENS}")

# Check the chunk size choice against the model's token limit
token_counts = np.array([len(embedder.tokenizer(PASSAGE_PREFIX + c["text"])["input_ids"]) for c in chunks])
print(f"Tokens per chunk: median {np.median(token_counts):.0f}, max {token_counts.max()}")
print(f"Chunks longer than the model limit (would be truncated): "
      f"{(token_counts > MAX_TOKENS).sum()} of {len(chunks)} ({(token_counts > MAX_TOKENS).mean():.1%})")

W1113 00:20:16.862000 12364 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model: BAAI/bge-m3 | dimension: 1024 | max tokens: 8192
Tokens per chunk: median 191, max 417
Chunks longer than the model limit (would be truncated): 0 of 8872 (0.0%)


In [11]:
t0 = time.time()
embeddings = embedder.encode(
    [PASSAGE_PREFIX + c["text"] for c in chunks],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"Embedded {len(chunks)} chunks -> {embeddings.shape} in {time.time() - t0:.1f}s")

Batches:   0%|          | 0/278 [00:00<?, ?it/s]

Embedded 8872 chunks -> (8872, 1024) in 198.5s


In [12]:
import chromadb

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# Rebuild from scratch so re-running the notebook never duplicates chunks
if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,                       # we supply our own embeddings
    configuration={"hnsw": {"space": "cosine"}},
)

BATCH = 1000
for start in range(0, len(chunks), BATCH):
    batch = chunks[start:start + BATCH]
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
        embeddings=embeddings[start:start + BATCH].tolist(),
    )

print(f"Stored {collection.count()} chunks in collection '{COLLECTION_NAME}' at {VECTOR_STORE_DIR}")

Stored 8872 chunks in collection 'agri_docs' at D:\agri-rag-assistant\backend\data\vector_store


## 2.4 Retrieval & Prompting

### Hybrid search in both languages
A first version searched with embeddings only and reached 70 % retrieval / 44 % correct answers.
Reading the failures showed three causes, each fixed here:

| What went wrong | Example | Fix |
|---|---|---|
| The embedding model doesn't know rare names | `التوتا أبسلوتا` (a tomato pest) retrieved cucumber and potato passages | **BM25 keyword search** fused with the embedding search |
| Crop names don't match across languages | *When should sugar beet be planted?* never matched `بنجر السكر` | the LLM **translates the question** (Arabic ↔ English); a fixed glossary forces correct crop names |
| A passage doesn't name its crop | the sunflower seed rate says `4 – 5 كجم بذرة / فدان`, not "sunflower" | **document topics** (2.2) are indexed with every chunk |

How a question is searched (all of it in `backend/app/services/retrieval.py`, used by the API too):
1. **Phrasings:** the question as written + its translation into the other language (`QUERY_TRANSLATION`).
2. For each phrasing, two ranked lists of 30 chunks: **embedding** similarity (Chroma) and **BM25** over
   normalised, light-stemmed words (`lexical.py`: Arabic article/suffixes removed, `ل ب ك ف` prefixes learned
   from the corpus, English Porter stemming). Glossary names are added to the keyword query: a crop's main
   name (`عباد الشمس` → `دوار الشمس`, the name in the topic labels) and other names of practices
   (`في أي وقت … زراعة` → `ميعاد الزراعة`, `موعد الزراعة`, the words the bulletins use).
3. **Reciprocal Rank Fusion:** a chunk scores Σ 1/(60 + rank) over the two lists of a phrasing; across
   phrasings it keeps its best score. RRF uses ranks only, so the two very different score scales never
   have to be compared. The best keyword match for the farmer's own wording always keeps a seat in the top-k
   (RRF alone can bury a passage only BM25 finds).
   **Crop seats:** when the question names a crop that has its own bulletin, that bulletin is also searched on
   its own and its best passages get 3 of the 6 seats (a general handbook or an Indian/FAO text about the
   same crop can otherwise fill every seat: "sugarcane setts per hectare" instead of the Egyptian "طن للفدان").
   A question comparing crops ("watermelon or tomato?") gives each crop 2 seats.
4. **Relevance gate:** the question is refused **without calling the LLM** unless its best cosine
   similarity over all phrasings reaches `MIN_SIMILARITY` (calibrated below) **or** it names a crop one of
   the documents is about (from the topic labels): a sunflower question is in scope when there is a
   sunflower bulletin, even if its wording scores low.

In [13]:
from app.services.generation import OllamaGenerator
from app.services.rag_service import RAGService
from app.services.retrieval import Retriever

generator = OllamaGenerator(host=OLLAMA_HOST, model=OLLAMA_MODEL, temperature=0.0, timeout=180,
                            think=OLLAMA_THINK, num_ctx=OLLAMA_NUM_CTX)
ollama_reachable, model_available = generator.status()
OLLAMA_OK = ollama_reachable and model_available
if not ollama_reachable:
    print(f"⚠️ Ollama is not reachable at {OLLAMA_HOST}. Start it (`ollama serve`) and re-run from here.")
elif not model_available:
    print(f"⚠️ Model '{OLLAMA_MODEL}' is not pulled. Run `ollama pull {OLLAMA_MODEL}` and re-run from here.")
else:
    print(f"✅ Ollama ready with {OLLAMA_MODEL}")
if QUERY_TRANSLATION and not OLLAMA_OK:
    print("   Without the LLM, questions are translated with glossary terms only.")

# Translations are cached so every experiment below searches with the same phrasings
_translate, _translations = generator.translate, {}
def cached_translate(question: str, target: str):
    if (question, target) not in _translations:
        _translations[(question, target)] = _translate(question, target)
    return _translations[(question, target)]
generator.translate = cached_translate

config = {"embedding_model": EMBEDDING_MODEL, "query_prefix": QUERY_PREFIX, "min_similarity": MIN_SIMILARITY}
t0 = time.time()
retriever = Retriever(collection, embedder, config, hybrid=HYBRID_SEARCH)  # builds the BM25 index
rag = RAGService(retriever, generator, top_k=TOP_K, min_similarity=MIN_SIMILARITY,
                 translate_queries=QUERY_TRANSLATION, self_check=SELF_CHECK)
print(f"Retriever ready in {time.time() - t0:.1f}s (hybrid={HYBRID_SEARCH}, top_k={TOP_K})")


def phrasings(question: str) -> list[str]:
    return rag.search_queries(question, detect_language(question))


for q in ["When should sugar beet be planted in Egypt?", "كيف أكافح حشرة التوتا أبسلوتا في الطماطم؟"]:
    queries = phrasings(q)
    print(f"\n{q}\n   searched as: {queries}")
    for name, hybrid in [("embeddings only", False), ("hybrid", True)]:
        print(f"   -- {name}")
        for c in retriever.search(queries, TOP_K, hybrid=hybrid).chunks:
            print(f"      {c.score:.3f}  {c.label:<48} {c.text[:60]!r}")

⚠️ Ollama is not reachable at http://localhost:11434. Start it (`ollama serve`) and re-run from here.
   Without the LLM, questions are translated with glossary terms only.
Retriever ready in 6.7s (hybrid=True, top_k=6)


Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download



When should sugar beet be planted in Egypt?
   searched as: ['When should sugar beet be planted in Egypt?', 'بنجر السكر ميعاد الزراعة']
   -- embeddings only
      0.759  1-نشرة-بنجر-السكر_compressed.pdf, p. 5           'زراعة وإنتاج بنجر السكر › ميعاد الزراعة\nيؤثر ميعاد الزراعة و'
      0.759  1-نشرة-بنجر-السكر_compressed.pdf, p. 5           'زراعة وإنتاج بنجر السكر › ميعاد الزراعة\nمرحلة النضج والحصاد '
      0.736  1-نشرة-بنجر-السكر_compressed.pdf, p. 16          'زراعة وإنتاج بنجر السكر › حصاد بنجر السكر\nيتم حصاد بنجر السك'
      0.711  1-نشرة-بنجر-السكر_compressed.pdf, p. 3           'زراعة وإنتاج بنجر السكر › مقدمة\nيمكن زراعة بنجر السكر فى جمي'
      0.702  1-نشرة-بنجر-السكر_compressed.pdf, p. 25          'زراعة وإنتاج بنجر السكر › ذبابة البنجر\nو تتغذي علي طبقة المي'
      0.693  NEW_sugarcane_seedlings.pdf, p. 8                'إنتاج شتلات قصب السكر › مقدمة › ميعاد الزراعة\nميعاد الزراعة '
   -- hybrid
      0.759  1-نشرة-بنجر-السكر_compressed.pdf, p. 5           'زراعة وإن

Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download



كيف أكافح حشرة التوتا أبسلوتا في الطماطم؟
   searched as: ['كيف أكافح حشرة التوتا أبسلوتا في الطماطم؟', 'tomato Tuta absoluta']
   -- embeddings only
      0.618  NEW_tomato.pdf, p. 32                            'نشرة الطماطم 2025 › ممارسات الزراعة الذكية مناخيا لإنتاج الط'
      0.559  NEW_tomato.pdf, pp. 16-17                        'نشرة الطماطم 2025 › العزيق ومكافحة الحشائش\nالحشائش من أكبر ا'
      0.541  NEW_tomato.pdf, pp. 23-24                        'نشرة الطماطم 2025 › العيوب الفسيولوجية › الوقاية من الإصابة\n'
      0.581  الباذنجان-17_compressed_compressed_compressed.pdf, p. 26 'الباذنجان › التداول والتخزين › المكافحة\nالري المتقارب وعدم ت'
      0.573  البطاطس_compressed.pdf, pp. 46-47                'زراعة وإنتاج البطاطس › المكافحة\n6 - ازالة النباتات المصابة ب'
      0.573  زراعة-وانتاج-الخيار.pdf, pp. 25-26               'زراعة وانتاج الخيار › الأمراض البكتيرية › المقاومة\n2- زراعة '
   -- hybrid
      0.618  NEW_tomato.pdf, p. 32                            'نشرة الطما

### Testing retrieval on the evaluation questions
`data/eval_questions.json` holds farming questions in Arabic and English (each with keywords a correct
passage should contain) plus out-of-scope questions that should **not** match anything.
A farming question counts as a retrieval hit when it passes the relevance gate and an expected keyword
appears in the `TOP_K` chunks given to the LLM (keywords match regardless of diacritics, alef/taa-marbuta
forms, digits and the Arabic article); an out-of-scope question counts as correct when the gate refuses it.

In [14]:
from app.services.grounding import contains_keyword

eval_questions = json.loads(EVAL_FILE.read_text(encoding="utf-8"))
print(f"{len(eval_questions)} questions "
      f"({sum(q['type'] == 'in_domain' for q in eval_questions)} in-domain, "
      f"{sum(q['type'] == 'out_of_scope' for q in eval_questions)} out-of-scope)")


def retrieval_relevant(q: dict, result) -> bool:
    passed = result.passes(MIN_SIMILARITY)
    if q["type"] == "out_of_scope":
        return not passed
    hits = [c for c in result.chunks if not q.get("expected_source") or q["expected_source"].lower() in c.source.lower()]
    return passed and contains_keyword(" ".join(c.text for c in hits), q["expected_keywords"])


search_by_q = {q["id"]: retriever.search(phrasings(q["question"]), TOP_K) for q in eval_questions}

88 questions (78 in-domain, 10 out-of-scope)


Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible.

### Calibrating the relevance threshold
Every embedding model has its own similarity scale, so a fixed number like 0.3 can't be reused across
models. We look at the best score each question gets: farming questions whose top-k really contains
the answer should score higher than out-of-scope questions. The threshold is placed where it
misclassifies the fewest of these questions (with the widest margin on ties); when the two groups
separate cleanly, that is the midpoint of the gap.

Questions that name a crop with its own document pass the gate anyway, so the threshold is calibrated on
the other farming questions (general ones such as conservation agriculture or soil testing) against the
out-of-scope questions.

*Caveat:* the threshold is tuned on the same questions we evaluate on, so the out-of-scope refusal rate
below is optimistic — add a few new off-topic questions to `eval_questions.json` to test it honestly.

In [15]:
def best_score(qid):
    return search_by_q[qid].best_similarity

answered = [q for q in eval_questions if q["type"] == "in_domain"
            and contains_keyword(" ".join(c.text for c in search_by_q[q["id"]].chunks), q["expected_keywords"])]
true_hits = [best_score(q["id"]) for q in answered if not search_by_q[q["id"]].named_crops] \
    or [best_score(q["id"]) for q in answered]  # (all of them if every question names a crop)
off_topic = [best_score(q["id"]) for q in eval_questions if q["type"] == "out_of_scope"]
print(f"Crop-named farming questions (pass the gate anyway): "
      f"{sum(bool(search_by_q[q['id']].named_crops) for q in answered)} of {len(answered)}")
print(f"Other farming questions with the answer in top-{TOP_K}: lowest best score = {min(true_hits, default=float('nan')):.3f}")
print(f"Out-of-scope questions:                      highest best score = {max(off_topic, default=float('nan')):.3f}")

def best_split(positives: list[float], negatives: list[float]) -> tuple[float, int]:
    """Threshold with the fewest errors (positive below it, negative at/above it); widest gap on ties."""
    scores = sorted(set(positives + negatives))
    candidates = [(a + b) / 2 for a, b in zip(scores, scores[1:])] or scores
    def errors(t):
        return sum(p < t for p in positives) + sum(n >= t for n in negatives)
    def margin(t):
        return min([abs(x - t) for x in positives + negatives])
    t = min(candidates, key=lambda t: (errors(t), -margin(t)))
    return round(t, 3), errors(t)


if CALIBRATE_THRESHOLD and true_hits and off_topic:
    MIN_SIMILARITY, n_errors = best_split(true_hits, off_topic)
    print(f"-> MIN_SIMILARITY calibrated to {MIN_SIMILARITY} "
          f"({n_errors} of {len(true_hits) + len(off_topic)} questions on the wrong side); saved to rag_config.json")
    if n_errors:
        print("   The groups overlap: some farming questions score no higher than off-topic ones. "
              "Look at their retrieved chunks in the table below.")
else:
    print(f"-> not calibrated (need farming questions with hits and out-of-scope questions): "
          f"keeping MIN_SIMILARITY = {MIN_SIMILARITY}")
rag.min_similarity = MIN_SIMILARITY  # the pipeline uses the calibrated gate from here on

Crop-named farming questions (pass the gate anyway): 54 of 72
Other farming questions with the answer in top-6: lowest best score = 0.485
Out-of-scope questions:                      highest best score = 0.553
-> MIN_SIMILARITY calibrated to 0.591 (1 of 28 questions on the wrong side); saved to rag_config.json
   The groups overlap: some farming questions score no higher than off-topic ones. Look at their retrieved chunks in the table below.


In [16]:
rows = []
for q in eval_questions:
    result = search_by_q[q["id"]]
    rows.append({
        "id": q["id"],
        "lang": detect_language(q["question"]),
        "type": q["type"],
        "question": q["question"],
        "top_source": result.chunks[0].label if result.chunks else "-",
        "best_similarity": result.best_similarity,
        "named_crops": ", ".join(result.named_crops),
        "passes_gate": result.passes(MIN_SIMILARITY),
        "retrieval_relevant": retrieval_relevant(q, result),
    })
retrieval_df = pd.DataFrame(rows)
in_dom = retrieval_df[retrieval_df.type == "in_domain"]
print(f"Retrieval hit rate (in-domain): {in_dom.retrieval_relevant.mean():.0%} | by language: "
      f"{in_dom.groupby('lang').retrieval_relevant.mean().map('{:.0%}'.format).to_dict()}")
retrieval_df

Retrieval hit rate (in-domain): 91% | by language: {'ar': '95%', 'en': '81%'}


,id,lang,type,question,top_source,best_similarity,named_crops,passes_gate,retrieval_relevant
0,q01,ar,in_domain,ما هو الميعاد المناسب لزراعة القطن في الأراضي القديمة؟,"نشرة-القطن2023_compressed-1.pdf, p. 7",0.7621,cotton,True,True
1,q02,ar,in_domain,ما أسباب ظاهرة الهياج الخضري في القطن؟,"نشرة-القطن2023_compressed-1.pdf, p. 13",0.7840,cotton,True,True
2,q03,ar,in_domain,ما هو أنسب ميعاد لزراعة القمح في الأراضي الجديدة؟,"NEW_wheat_new.pdf, pp. 6-7",0.7358,wheat,True,True
3,q04,ar,in_domain,كم كيلو تقاوي قمح يحتاج الفدان تحت نظام الري المحوري (البيفوت)؟,"NEW_wheat_new.pdf, p. 19",0.7183,,True,True
4,q05,ar,in_domain,ما معدل تقاوي القمح عند الزراعة بالنقر على المصاطب؟,"NEW_wheat_new.pdf, pp. 10-11",0.6924,wheat,True,True
...,...,...,...,...,...,...,...,...,...
83,q84,en,out_of_scope,How do I change a flat tyre on my car?,"farmerbook.pdf, p. 183",0.4871,,False,True
84,q85,ar,out_of_scope,ما هو أفضل هاتف ذكي يمكن شراؤه هذا العام؟,"NEW_wheat_new.pdf, p. 49",0.3995,,False,True
85,q86,en,out_of_scope,Can you recommend a good recipe for chocolate cake?,"917873240-Key-Fact.pdf, p. 64",0.4805,,False,True
86,q87,ar,out_of_scope,كيف أتعلم البرمجة بلغة بايثون؟,"AGRI_013.pdf, p. 73",0.4490,,False,True


### Which search method finds the answer? (retrieval experiment)
The same questions, searched four ways; the table shows how often a passage containing the answer is
among the chunks given to the LLM (`@TOP_K`) and among the first 10. *Embeddings, question only* is the
first version of this project; the last row is what the API uses.

In [17]:
in_domain_qs = [q for q in eval_questions if q["type"] == "in_domain"]
methods = {
    "embeddings, question only (first version)": dict(translate=False, hybrid=False),
    "embeddings + translation": dict(translate=True, hybrid=False),
    "BM25 keywords + translation": dict(translate=True, hybrid="bm25"),
    "hybrid (embeddings + BM25) + translation": dict(translate=True, hybrid=True),
}


def bm25_only(queries: list[str], k: int) -> list:
    """Keyword search alone, phrasings fused the same way (best rank across phrasings)."""
    best = {}
    for query in queries:
        for rank, (i, _) in enumerate(retriever.keyword_search(query, 30)):
            best[i] = max(best.get(i, 0.0), 1 / (61 + rank))
    return [retriever._texts[i] for i in sorted(best, key=best.get, reverse=True)[:k]]


comparison, misses = [], {}
for name, m in methods.items():
    hits = {k: 0 for k in (TOP_K, 10)}
    for q in in_domain_qs:
        queries = phrasings(q["question"]) if m["translate"] else [q["question"]]
        for k in hits:
            if m["hybrid"] == "bm25":
                texts = bm25_only(queries, k)
            else:
                texts = [c.text for c in retriever.search(queries, k, hybrid=m["hybrid"]).chunks]
            found = contains_keyword(" ".join(texts), q["expected_keywords"])
            hits[k] += found
            if k == TOP_K and not found:
                misses.setdefault(name, []).append(q["id"])
    comparison.append({"method": name, f"answer in top {TOP_K}": f"{hits[TOP_K] / len(in_domain_qs):.0%}",
                       "answer in top 10": f"{hits[10] / len(in_domain_qs):.0%}"})
display(pd.DataFrame(comparison))
print("Still missed by the hybrid search:", misses.get("hybrid (embeddings + BM25) + translation", []))

,method,answer in top 6,answer in top 10
0,"embeddings, question only (first version)",96%,96%
1,embeddings + translation,95%,95%
2,BM25 keywords + translation,83%,91%
3,hybrid (embeddings + BM25) + translation,92%,94%


Still missed by the hybrid search: ['q53', 'q55', 'q64', 'q72', 'q74', 'q75']


### Egyptian Arabic questions
`data/eval_questions_egyptian.json` asks the same facts as q01–q55 the way farmers speak (*"إمتى أزرع الأوطة؟"*).
The API rewrites Egyptian words into standard Arabic before searching (`backend/app/services/dialect.py`); with
`models/bge-m3-egyptian` the embeddings also understand what the word list misses. Same search as the API
(hybrid + translation), with and without the rewrite.

In [18]:
from app.services.dialect import to_msa

EGYPTIAN_EVAL_FILE = PROJECT_ROOT / "data" / "eval_questions_egyptian.json"
egyptian_questions = json.loads(EGYPTIAN_EVAL_FILE.read_text(encoding="utf-8")) if EGYPTIAN_EVAL_FILE.exists() else []
egyptian_rows, egyptian_misses = [], []
for label, prepare in [("as typed", lambda text: text), ("after the dialect rewrite (what the API searches)", to_msa)]:
    found = 0
    for q in egyptian_questions:
        hit = retrieval_relevant(q, retriever.search(phrasings(prepare(q["question"])), TOP_K))
        found += hit
        if not hit and prepare is to_msa:
            egyptian_misses.append(q["id"])
    egyptian_rows.append({"Egyptian questions": label,
                          f"answer in top {TOP_K}": f"{found / max(len(egyptian_questions), 1):.0%}"})
print(f"{len(egyptian_questions)} Egyptian questions | embedding model: {EMBEDDING_MODEL}")
print("Missed after the rewrite:", egyptian_misses)
pd.DataFrame(egyptian_rows)

Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
Query translation failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible.

55 Egyptian questions | embedding model: BAAI/bge-m3
Missed after the rewrite: ['e02', 'e09', 'e15', 'e30', 'e39', 'e43', 'e44', 'e47', 'e50', 'e52', 'e53', 'e55']


,Egyptian questions,answer in top 6
0,as typed,64%
1,after the dialect rewrite (what the API searches),78%


### Prompt template and citation-style grounding
The prompt (defined once in `backend/app/services/generation.py`) numbers each retrieved passage
`[1] … [k]` with its file and page, and instructs the model to:
- answer **only** from the passages, citing them as `[n]`;
- answer in the **language of the question** (detected from its script), translating facts from
  passages in the other language faithfully — the LLM is `qwen2.5`, which handles Arabic and English;
- start with the direct answer and never repeat the question;
- copy numbers, doses, dates and units exactly — never calculate or guess one;
- reply with a fixed refusal sentence (English or Arabic) when none of the passages answers;
- keep it short and practical. A one-line example shows the expected `[n]` style.

The rules are short and numbered on purpose: small local models (3–7 B) follow them better than prose.
Each passage header names its file, page and topic. `num_ctx` is set to 8192 tokens: Ollama's default
window is smaller than six Arabic passages plus the rules, and Ollama silently cuts the *start* of an
over-long prompt — the rules.

### Answer self-check (`backend/app/services/grounding.py` + `rag_service.py`)
Small models make a few typical mistakes. Every answer is checked before the farmer sees it:

| Problem | Example from our runs | What happens |
|---|---|---|
| citation to a passage that doesn't exist | `[6]` with 4 passages | removed |
| refusal followed by the answer | "I don't have enough information… However, according to [1], …" | the refusal is cut, the answer kept |
| sentence marked with the wrong passage | Tuta answer copied from the tomato bulletin but marked `[1]` (Key-Facts) | re-pointed to the passage that contains its words |
| fact copied from a passage without `[n]` | mango answer with no citation | the passage sharing most of its words is cited |
| the question repeated as the answer | `كم كيلو تقاوي يحتاج فدان الفول البلدي؟ [1]` | the LLM gets its answer back with **every** problem named and tries **once** more; the new answer is kept only if it is better |
| wrong language | English maize question answered with the Arabic passage copied | same retry ("write it in English") |
| passage about another crop | olive-oil harvest answered from a citrus passage; faba-bean seed rate from the soybean bulletin | same retry ("answer only from passages about olives") |
| a number not in the passages | "40 to 60 t/ha" when the passage says 20–30 | same retry; if the number survives, its sentence is removed |
| sentences not in the passages | resistance answer with < 60 % of its words in the passages | same retry; uncited sentences without passage words are removed |
| refusal although passages were found | the arable-land question (refused twice with 5 passages) | a second look at only the **2 passages sharing most words with the question**; kept only if it passes the grounding check |

For a question in one language and passages in the other, the prompt also shows the translated question
(the one the search used), so a small model can match `seed drill` to `التسطير`.

Pesticide answers get a fixed safety line (label + protective equipment) added by the code, not by the LLM.
The `[n]` markers are mapped back to file + page, which become the `sources` returned by the API.

In [19]:
from app.services.generation import FEEDBACK, USER_PROMPT_TEMPLATE, build_messages, build_system_prompt
from app.services.grounding import is_refusal

print("SYSTEM PROMPT (English question)\n" + "-" * 60 + "\n" + build_system_prompt("en"))
print("\nSYSTEM PROMPT (Arabic question) — only the language and refusal sentence change\n" + "-" * 60)
print("\n".join(l for l in build_system_prompt("ar").splitlines() if "Arabic" in l or "reply with exactly" in l))
print("\nUSER PROMPT TEMPLATE\n" + "-" * 60 + "\n" + USER_PROMPT_TEMPLATE)
print("\nRETRY MESSAGES (sent with the first answer when the self-check finds a problem)\n" + "-" * 60)
for problem, text in FEEDBACK.items():
    print(f"{problem:>8}: {text}")

SYSTEM PROMPT (English question)
------------------------------------------------------------
You are SAQYA (ساقية), a farming assistant. You answer a farmer's question using ONLY the numbered passages from agricultural documents that come with it.

Rules:
1. Write the answer in English, even when the passages are in another language.
2. Start with the direct answer. Do not repeat the question.
3. End every sentence with the number of the passage it comes from, like [1] or [2][3].
4. Copy numbers, quantities, doses, dates and units exactly as the passages write them. Never calculate, convert or guess a number.
5. Say only what the passages say. If none of the passages answers the question, reply with exactly: "I don't have enough information in my documents to answer that."
6. If the question compares two or more things, first give what the passages say about each one (with its [n]), then compare them.
7. Keep it short and practical: one to five sentences or bullet points.

Example:
Pa

In [20]:
# What the LLM actually receives for one question
demo_question = eval_questions[0]["question"]
demo_messages = build_messages(demo_question, search_by_q[eval_questions[0]["id"]].chunks)
print(demo_messages[1]["content"][:1500])

Passages:

[1] (source: نشرة-القطن2023_compressed-1.pdf, p. 7; about: القطن / cotton)
زراعة وانتاج القطن المصري › خدمة و زراعة القطن في الأراضي القديمة ميعاد الزراعة المناسب
زيادة المحصول وجودة رتبته وزيادة تصافى الحليج.
الحد من الإصابة بالآفات والهروب منها خاصة ديدان اللوز والحشرات الثاقبة الماصة فى نهاية الموسم.
المحافظة على صفات التيلة المميزة للصنف. جنى المحصول مبكرا مما يتيح فرصة لزراعة المحاصيل الشتوية فى مواعيدها. الزراعة فى الميعاد المناسب من أهم عناصر المكافحة المتكاملة حيث يمكن تجنب الضرر الناتج عن الآفات بالزراعة في وقت مناسب ، ولذلك فإن اختاف مواعيد الزراعة في الحقول المتقاربة والترقيع المتأخر يساعدان علي هجرة الحشرات وانتشارها بين الحقول المصابة والسليمة.

[2] (source: نشرة-القطن2023_compressed-1.pdf, p. 7; about: القطن / cotton)
زراعة وانتاج القطن المصري › خدمة و زراعة القطن في الأراضي القديمة ميعاد الزراعة المناسب
خال شهري )مارس وإبريل( عند توافر الظروف الجوية المناسبة ويعتمد ميعاد الزراعة أساسا على درجة حرارة التربة ويجب الزراعة عند ثبات درجة حرارة التربة عند 15مo لمدة 

In [21]:
# One Arabic and one English farming question from the evaluation set, through the API's pipeline
demo_questions = [
    next((x["question"] for x in eval_questions if x["type"] == "in_domain" and detect_language(x["question"]) == lang), None)
    for lang in ("ar", "en")
]
if not OLLAMA_OK:
    print("LLM not available — start Ollama and re-run from the 'Hybrid search' cell.")
for q in demo_questions:
    if q and OLLAMA_OK:
        r = rag.answer(q)
        note = f"\n\n*self-check: {r.problem}, retried={r.retried}*" if r.problem else ""
        display(Markdown(f"**Q:** {q}\n\n**A:** {r.answer}\n\n**Sources:** {', '.join(r.sources) or '—'}{note}"))

LLM not available — start Ollama and re-run from the 'Hybrid search' cell.


## 2.5 Vision Component

**Not used — this project follows the Core Track (text-only RAG).**

*Extended Track idea for later:* a farmer uploads a photo of a sick leaf; a YOLOv8 classification model
fine-tuned on a plant-disease image set (e.g. PlantVillage) predicts the crop and disease
(`tomato — early blight, 0.91`). That label is (1) added to the retrieval query so the right
disease-management passages are found, and (2) inserted into the prompt as
`Image analysis: tomato early blight (confidence 0.91)` so the answer can refer to it — still citing
only the documents for any treatment advice.

## 2.6 Evaluation

For each test question we record the top retrieved source, the generated answer, and three judgements:

| Column | How it is decided |
|---|---|
| `retrieval_relevant` | in-domain: passes the relevance gate and an expected keyword appears in the chunks given to the LLM (from `expected_source`, if set) · out-of-scope: refused by the gate |
| `grounded` | the answer cites at least one passage, ≥ 60% of its content words appear in the passages (skipped when the answer translates passages in another language), and **every number in the answer appears in the passages** (catches invented doses; Arabic-Indic digits and right-to-left ranges are normalised) · refusals count as grounded. Same function the API's self-check uses. |
| `correct` | in-domain: not a refusal, grounded, and contains an expected keyword (`answer_keywords` if the question has them, else `expected_keywords`) · out-of-scope: the assistant refused |
| `problem` / `retried` | what the self-check found in the LLM's first answer, and whether it asked the LLM again |

Every question goes through `rag.answer()` — the exact code behind `POST /query`.
The automatic labels are a first pass. **Read each answer** and fix any wrong label in `MANUAL_OVERRIDES`.

*Question set:* `data/eval_questions.json` was rewritten for the cleaned corpus: 78 farming questions
(field crops, vegetables, fruit trees, general topics and 11 cross-language ones, where the question and the
document are in different languages) and 10 out-of-scope ones. Every `expected_keywords` entry was checked
against the indexed chunks, and `expected_source` names the bulletin the answer must come from. The
`category` field is used for the per-category rates in the summary. The previous 32-question set (used for
runs 1–2) is kept in `data/eval_questions_v1.json`; set `EVAL_FILE` to it to compare like with like.

*Label audit (runs 1–2, old question ids):* some keywords were narrower than their source passage, and were
widened to what the passage itself says: q10 (the palm bulletin recommends planting "from spring to the hot summer months" as
well as "any time except December and January"), q24 (AGRI_013 lists integrated pest management and lower
selection pressure as the first requirements, alongside rotating pesticide groups and mixtures; its
section "تكتيكات إدارة السيطرة على مقاومة الآفات" is an expected passage too) and q09 (`العام التال` also
matches "the next year" written `التالٍ`).

In [22]:
from app.services.grounding import grounding_check


def auto_correct(q: dict, answer: str, grounded: bool) -> bool:
    if q["type"] == "out_of_scope":
        return is_refusal(answer)
    keywords = q.get("answer_keywords") or q["expected_keywords"]
    return (not is_refusal(answer)) and grounded and contains_keyword(answer, keywords)


# After reading the answers below, correct any automatic label here, e.g. {"q04": True, "q09": False}
MANUAL_OVERRIDES: dict[str, bool] = {}

eval_rows, details = [], {}
for q in eval_questions:
    result = search_by_q[q["id"]]
    row = {
        "id": q["id"],
        "lang": detect_language(q["question"]),
        "type": q["type"],
        "question": q["question"],
        "retrieved_source": result.chunks[0].label if result.passes(MIN_SIMILARITY) and result.chunks
        else "— (nothing relevant)",
        "best_similarity": result.best_similarity,
        "retrieval_relevant": retrieval_relevant(q, result),
    }
    if OLLAMA_OK:
        t0 = time.time()
        r = rag.answer(q["question"])
        check = grounding_check(r.answer, r.chunks, q["question"])
        correct = auto_correct(q, r.answer, check["grounded"])
        row.update({
            "answer": r.answer,
            "sources": "; ".join(r.sources),
            "grounded": check["grounded"],
            "support": check["support"],
            "unsupported_numbers": ", ".join(check["unsupported_numbers"]),
            "correct": MANUAL_OVERRIDES.get(q["id"], correct),
            "problem": r.problem or "",
            "retried": r.retried,
            "first_answer": r.first_answer,
            "latency_s": round(time.time() - t0, 1),
        })
        details[q["id"]] = r.chunks
    eval_rows.append(row)

eval_df = pd.DataFrame(eval_rows)
if not OLLAMA_OK:
    print("⚠️ LLM not available: only the retrieval columns are filled. Start Ollama and re-run from 2.4.")
eval_df[[c for c in ["id", "lang", "question", "retrieved_source", "retrieval_relevant", "answer", "grounded",
                     "correct", "problem", "retried"] if c in eval_df]]

⚠️ LLM not available: only the retrieval columns are filled. Start Ollama and re-run from 2.4.


,id,lang,question,retrieved_source,retrieval_relevant
0,q01,ar,ما هو الميعاد المناسب لزراعة القطن في الأراضي القديمة؟,"نشرة-القطن2023_compressed-1.pdf, p. 7",True
1,q02,ar,ما أسباب ظاهرة الهياج الخضري في القطن؟,"نشرة-القطن2023_compressed-1.pdf, p. 13",True
2,q03,ar,ما هو أنسب ميعاد لزراعة القمح في الأراضي الجديدة؟,"NEW_wheat_new.pdf, pp. 6-7",True
3,q04,ar,كم كيلو تقاوي قمح يحتاج الفدان تحت نظام الري المحوري (البيفوت)؟,"NEW_wheat_new.pdf, p. 19",True
4,q05,ar,ما معدل تقاوي القمح عند الزراعة بالنقر على المصاطب؟,"NEW_wheat_new.pdf, pp. 10-11",True
...,...,...,...,...,...
83,q84,en,How do I change a flat tyre on my car?,— (nothing relevant),True
84,q85,ar,ما هو أفضل هاتف ذكي يمكن شراؤه هذا العام؟,— (nothing relevant),True
85,q86,en,Can you recommend a good recipe for chocolate cake?,— (nothing relevant),True
86,q87,ar,كيف أتعلم البرمجة بلغة بايثون؟,— (nothing relevant),True


In [23]:
def rate(series) -> str:
    s = series.dropna()
    return f"{s.astype(bool).mean():.0%} ({int(s.astype(bool).sum())}/{len(s)})" if len(s) else "n/a"


ind = eval_df[eval_df.type == "in_domain"]
oos = eval_df[eval_df.type == "out_of_scope"]
have_llm = "correct" in eval_df
summary = {
    "Questions evaluated": len(eval_df),
    "Retrieval relevant (in-domain)": rate(ind.retrieval_relevant),
    "Out-of-scope refused by the gate": rate(oos.retrieval_relevant),
}
if have_llm:
    summary.update({
        "Answers grounded (all)": rate(eval_df.grounded),
        "Answers correct (in-domain)": rate(ind.correct),
        "Out-of-scope correctly refused": rate(oos.correct),
        "Correct, Arabic questions (in-domain)": rate(ind[ind.lang == "ar"].correct),
        "Correct, English questions (in-domain)": rate(ind[ind.lang == "en"].correct),
        "First answers fixed by the self-check": f"{int((eval_df.problem != '').sum())} found, "
                                                  f"{int(eval_df.retried.sum())} retried",
        "Mean latency per question (s)": round(eval_df.latency_s.mean(), 1),
    })
# per category (field crops, vegetables, fruit trees, general, cross-language), from eval_questions.json
categories = {q["id"]: q.get("category", "") for q in eval_questions}
if any(categories.values()):
    eval_df["category"] = eval_df.id.map(categories)
    ind = eval_df[eval_df.type == "in_domain"]
    for cat, group in ind.groupby("category", sort=False):
        summary[f"Retrieval relevant, {cat}"] = rate(group.retrieval_relevant)
        if have_llm:
            summary[f"Correct, {cat}"] = rate(group.correct)
summary.update({"Retrieval": f"{'hybrid (' + EMBEDDING_MODEL.split('/')[-1] + ' + BM25)' if HYBRID_SEARCH else 'embeddings only'}, top_k={TOP_K}",
                "LLM": OLLAMA_MODEL if OLLAMA_OK else "not available"})
summary_df = pd.DataFrame(summary.items(), columns=["metric", "value"])
summary_df

,metric,value
0,Questions evaluated,88
1,Retrieval relevant (in-domain),91% (71/78)
2,Out-of-scope refused by the gate,100% (10/10)
3,"Retrieval relevant, field crops",100% (23/23)
4,"Retrieval relevant, vegetables",100% (19/19)
5,"Retrieval relevant, fruit trees",100% (9/9)
6,"Retrieval relevant, general",81% (13/16)
7,"Retrieval relevant, cross-language",64% (7/11)
8,Retrieval,"hybrid (bge-m3 + BM25), top_k=6"
9,LLM,not available


In [24]:
# Save the results: CSV for analysis, Markdown table to paste into the README
def md_table(df: pd.DataFrame) -> str:
    def cell(v):
        if isinstance(v, bool) or v is None:
            return {True: "✅", False: "❌", None: "—"}[v]
        return str(v).replace("|", "\\|").replace("\n", " ")
    header = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return header + "\n".join("| " + " | ".join(cell(v) for v in row) + " |" for row in df.itertuples(index=False))


eval_df.to_csv(RESULTS_DIR / "eval_results.csv", index=False, encoding="utf-8-sig")  # opens in Excel with Arabic
table = eval_df[[c for c in ["id", "lang", "question", "retrieved_source", "answer", "grounded", "correct"] if c in eval_df]].copy()
if "answer" in table:
    table["answer"] = table.answer.str.slice(0, 160) + np.where(table.answer.str.len() > 160, "…", "")
(RESULTS_DIR / "eval_results.md").write_text(
    "### Summary\n\n" + md_table(summary_df) + "\n\n### Per question\n\n" + md_table(table) + "\n",
    encoding="utf-8",
)
print("Saved", RESULTS_DIR / "eval_results.csv", "and", RESULTS_DIR / "eval_results.md")

Saved D:\agri-rag-assistant\notebooks\results\eval_results.csv and D:\agri-rag-assistant\notebooks\results\eval_results.md


In [25]:
# Inspect the failures: what was retrieved, and what the model said
flags = eval_df.retrieval_relevant == False
if "correct" in eval_df:
    flags |= (eval_df.grounded == False) | (eval_df.correct == False)
problems = eval_df[flags]
if problems.empty:
    print("No failures flagged by the automatic checks — still read the answers above yourself.")
for _, row in problems.iterrows():
    print(f"=== {row.id}: {row.question}")
    print(f"retrieval_relevant={row.retrieval_relevant} best_similarity={row.best_similarity}")
    if "correct" in eval_df:
        print(f"grounded={row.grounded} correct={row.correct} support={row.support} "
              f"unsupported_numbers=[{row.unsupported_numbers}] problem={row.problem or '-'} retried={row.retried}")
        print("ANSWER:", row.answer[:400])
        if row.retried:
            print("FIRST ANSWER:", row.first_answer[:300])
    for c in details.get(row.id) or search_by_q[row.id].chunks:
        print(f"   {c.score:.3f} {c.label}: {c.text[:120]!r}")
    print()

# The self-check at work: first answers that had a problem, and what the farmer finally got
if "problem" in eval_df:
    fixed = eval_df[eval_df.problem != ""][["id", "problem", "retried", "first_answer", "answer", "correct"]]
    display(fixed)

=== q53: كيف أمنع الآفات من اكتساب مقاومة للمبيدات؟
retrieval_relevant=False best_similarity=0.6994
   0.677 AGRI_011_PDF.pdf, p. 17: 'The International Code of Conduct on Pesticide Management › Pesticide management\n3.7 All relevant entities addressed by '
   0.667 AGRI_013.pdf, pp. 78-79: 'مدونة السلوك الدولية بشأن توزيع المبيدات واستعمالها › الخطوط التوجيهية الخاصة بمنع وإدارة المقاومة لفعل مبيدات الآفات › '
   0.668 AGRI_013.pdf, pp. 78-79: 'مدونة السلوك الدولية بشأن توزيع المبيدات واستعمالها › الخطوط التوجيهية الخاصة بمنع وإدارة المقاومة لفعل مبيدات الآفات › '
   0.654 farmerbook.pdf, pp. 47-48: 'Farmer’s Handbook on Basic Agriculture › Good Agricultural Practices (GAP) › Privilege resistant species The benefits of'
   0.654 AGRI_013.pdf, p. 46: 'مدونة السلوك الدولية بشأن توزيع المبيدات واستعمالها › الخطوط التوجيهية الخاصة بمنع وإدارة المقاومة لفعل مبيدات الآفات › '
   0.608 AGRI_013.pdf, pp. 38-39: 'مدونة السلوك الدولية بشأن توزيع المبيدات واستعمالها › الخطوط التوجيهية الخاصة بمن

### 2.6b Does the chunk size matter? (retrieval only, no LLM)
Re-chunks the corpus at three sizes and measures the in-domain retrieval hit rate@k for each,
using in-memory cosine similarity. This backs up the 800-character choice in 2.2 with numbers from
*your* documents.

In [26]:
RUN_CHUNK_ABLATION = True  # set False to skip if your corpus is very large

if RUN_CHUNK_ABLATION:
    in_domain_qs = [q for q in eval_questions if q["type"] == "in_domain"]
    # original questions, embeddings only, so the comparison isolates the effect of chunk size
    q_vecs = embedder.encode([QUERY_PREFIX + q["question"] for q in in_domain_qs], normalize_embeddings=True)
    ablation = []
    for size, overlap in [(400, 80), (800, 150), (1200, 200)]:
        alt, _ = drop_duplicate_chunks(chunk_corpus(clean_pages, size, overlap))
        vecs = embedder.encode([PASSAGE_PREFIX + c["text"] for c in alt], batch_size=32, normalize_embeddings=True)
        sims = q_vecs @ vecs.T
        hit = []
        for qi, q in enumerate(in_domain_qs):
            top = np.argsort(-sims[qi])[:TOP_K]
            hit.append(contains_keyword(" ".join(alt[i]["text"] for i in top), q["expected_keywords"]))
        tokens = [len(embedder.tokenizer(PASSAGE_PREFIX + c["text"])["input_ids"]) for c in alt]
        ablation.append({
            "chunk_size": size, "overlap": overlap, "chunks": len(alt),
            f"hit_rate@{TOP_K}": f"{np.mean(hit):.0%}",
            "mean_top1_similarity": round(float(np.mean(sims.max(axis=1))), 3),
            "truncated_chunks": f"{np.mean(np.array(tokens) > MAX_TOKENS):.0%}",
        })
    display(pd.DataFrame(ablation))

,chunk_size,overlap,chunks,hit_rate@6,mean_top1_similarity,truncated_chunks
0,400,80,16947,92%,0.704,0%
1,800,150,8872,92%,0.700,0%
2,1200,200,6052,90%,0.699,0%


### Failure cases and mitigations ✏️ *(add what you see in the failure printout above)*

Run 1 (embeddings only, 4 chunks, no checks): **70 % retrieval, 44 % correct**.
Run 2 (hybrid search, topics, self-check): **89 % retrieval, 67 % correct**.
Run 3 (cleaned corpus in `data/clean/`, section-aware chunks with heading paths, new question set): ✏️ fill in from 2.6.

What failed in runs 1–2, and the fix (question ids refer to the old set in `data/eval_questions_v1.json`):

| Failure (question) | Cause | Mitigation now in the pipeline |
|---|---|---|
| Tuta absoluta (q07), sunflower seed rate (q06) | rare names / passage doesn't name the crop | BM25 keyword search fused with embeddings; document topics |
| Sugar beet, maize hybrids in English (q13, q15) | names don't match across languages; the LLM translated "sugar beet" as kiwi in Chinese | query translation + crop/practice glossary; foreign-script translations rejected |
| Question echoed (q03), refusal + answer (q16), no citation (q09), citation `[6]` (q17) | small-model output mistakes | LLM-free repairs + one retry with the problem named |
| "40–60 t/ha" when the bulletin says 20–30 (q19) | invented number | number check → retry → sentence removed if still unsupported |
| Refused although the passage was found (q20) | over-cautious model | one second look, kept only if grounded |
| Prompt rules ignored at times | Ollama's default context window cut the start of long Arabic prompts | `num_ctx = 8192` |
| Out-of-scope questions (World Cup, beekeeping, tilapia…) | — | relevance gate calibrated on bge-m3 scores: refused without calling the LLM |
| *Run 2:* sunflower seed rate refused by the gate (0.568 < 0.59) | the bulletin says عباد الشمس, the farmer دوار الشمس | gate also passes questions naming a crop that has a document |
| *Run 2:* palm planting time (q10): five chunks of one English paper | "في أي وقت … زراعة" didn't match "موعد الزراعة" | Arabic planting-time phrasing in the glossary → keyword query gets `موعد/ميعاد الزراعة` |
| *Run 2:* "60 kg/ha [3]" replaced by a refusal (q14) | bug: an answer of numbers only counted as empty | numbers count as content; translated question shown in the prompt |
| *Run 2:* English maize answer in Arabic (q15), olive answer from a citrus passage (q17), faba seed rate from the soybean bulletin (q03) | small-model mistakes | language and crop checks → retry |

Known limits:
- **Tables:** a value read across a flattened table row (q23, citrus fertiliser schedule) depends on the
  extracted row order; two citrus bulletins with different schedules make the question ambiguous.
- **Model size:** `qwen2.5:3b` still paraphrases loosely in Arabic. `qwen2.5:7b` (≈5 GB, `OLLAMA_MODEL=qwen2.5:7b`)
  follows the rules noticeably better if the GPU has the memory.
- The gate is calibrated on the same out-of-scope questions it is tested on: add new off-topic questions
  to check it honestly.

*Observed in my run:* ___

## 2.7 Export

The Chroma database is already persisted in `backend/data/vector_store/` (it was created there in 2.3).
Here we write `rag_config.json` next to it so the backend knows which embedding model and collection
to load, then reopen the store from disk exactly as the backend does to prove it works without
rebuilding.

In [27]:
rag_config = {
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": EMBEDDING_DIM,
    "query_prefix": QUERY_PREFIX,
    "passage_prefix": PASSAGE_PREFIX,
    "distance": "cosine",
    "normalize_embeddings": True,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "chunking": "section-aware with heading path (cleaned PDFs) + sentence-aware fixed-size with overlap",
    "documents_dir": DOCS_DIR.relative_to(PROJECT_ROOT).as_posix(),
    "structured_documents": int(inventory_df.structured.sum()),
    "section_chunks": int((chunk_df.section != "").sum()),
    "top_k": TOP_K,
    "retrieval": "hybrid: embeddings + BM25, reciprocal rank fusion" if HYBRID_SEARCH else "embeddings",
    "hybrid_search": HYBRID_SEARCH,
    "query_translation": QUERY_TRANSLATION,
    "self_check": SELF_CHECK,
    "min_similarity": MIN_SIMILARITY,
    "llm_model": OLLAMA_MODEL,
    "languages": chunk_df.lang.value_counts().to_dict(),
    "num_documents": int(chunk_df.source.nunique()),
    "repeated_sentences_skipped": int(sum(repeated_sentences.values())),
    "repeated_chunks_dropped": int(sum(dropped_chunks.values())),
    "unreadable_pages_skipped": int(sum(unreadable_pages.values())),
    "num_chunks": len(chunks),
    "documents": sorted(chunk_df.source.unique().tolist()),
    "document_topics": {k: v for k, v in sorted(topics.items()) if v},
    "document_titles": {k: v for k, v in sorted(chunk_df.groupby("source").title.first().items()) if v},
    "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(VECTOR_STORE_DIR / "rag_config.json").write_text(json.dumps(rag_config, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(rag_config, indent=2, ensure_ascii=False))

{
  "collection_name": "agri_docs",
  "embedding_model": "BAAI/bge-m3",
  "embedding_dim": 1024,
  "query_prefix": "",
  "passage_prefix": "",
  "distance": "cosine",
  "normalize_embeddings": true,
  "chunk_size": 800,
  "chunk_overlap": 150,
  "chunking": "section-aware with heading path (cleaned PDFs) + sentence-aware fixed-size with overlap",
  "documents_dir": "data/clean",
  "structured_documents": 52,
  "section_chunks": 8491,
  "top_k": 6,
  "retrieval": "hybrid: embeddings + BM25, reciprocal rank fusion",
  "hybrid_search": true,
  "query_translation": true,
  "self_check": true,
  "min_similarity": 0.591,
  "llm_model": "qwen2.5:3b",
  "languages": {
    "ar": 5431,
    "en": 3441
  },
  "num_documents": 52,
  "repeated_sentences_skipped": 2167,
  "repeated_chunks_dropped": 145,
  "unreadable_pages_skipped": 4,
  "num_chunks": 8872,
  "documents": [
    "1-2.pdf",
    "1-نشرة-بنجر-السكر_compressed.pdf",
    "135905761430.pdf",
    "2-البحث-الثاني.pdf",
    "2.pdf",
    "57713

In [28]:
# Reopen from disk with a fresh client — exactly what the backend does at startup
fresh = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR)).get_collection(COLLECTION_NAME, embedding_function=None)
assert fresh.count() == len(chunks), "persisted store does not match the chunks"

api_retriever = Retriever.from_store(VECTOR_STORE_DIR, embedder=embedder, hybrid=HYBRID_SEARCH)
q = eval_questions[0]["question"]
top = api_retriever.search([q], 1).chunks[0]
print(f"Reloaded {api_retriever.count()} chunks as the API does. Top hit for {q!r}: {top.label} (topic: {top.topic or '-'})")

size_mb = sum(f.stat().st_size for f in VECTOR_STORE_DIR.rglob("*") if f.is_file()) / 1024 ** 2
print(f"Vector store size: {size_mb:.1f} MB -> "
      + ("small enough to commit (< 50 MB)" if size_mb < 50 else "too large to commit: keep it in .gitignore and rebuild with this notebook"))

Reloaded 8872 chunks as the API does. Top hit for 'ما هو الميعاد المناسب لزراعة القطن في الأراضي القديمة؟': نشرة-القطن2023_compressed-1.pdf, p. 7 (topic: القطن / cotton)
Vector store size: 412.6 MB -> too large to commit: keep it in .gitignore and rebuild with this notebook


**Done.** Start the backend (`cd backend && uvicorn app.main:app --reload`) — it loads
`backend/data/vector_store/` once at startup and serves `POST /query`.
Paste `notebooks/results/eval_results.md` into the README's evaluation section.